# CV Detection Pipeline — The Naylor Model

Measures **pitcher delivery time** (first move → ball release) from Statcast broadcast clips
using YOLOv8-pose, then combines it with Savant per-attempt lead distances to compute a
**runner closing-velocity metric** (`avg_velocity_ftps = gain_to_release_ft / delivery_s`).

**Pipeline:**
1. Fetch per-attempt leads from Savant  (`fetch_leads.py`)
2. Download broadcast clips             (`fetch_clips.py`)
3. Extract delivery times via YOLO      (`extract_delivery.py`)
4. Build the per-attempt delivery table (`build_delivery.py`)
5. Aggregate by pitcher                 (`aggregate_delivery.py`)
6. Compute runner velocity metric       (`delivery_velocity.py`)
7. Statcast reference cross-check       (`statcast_ref.py`)
8. Evaluate accuracy vs manual labels   (`evaluate.py`)

**Tip:** Use `run_runner.py` to run steps 1–3 in one command.
**Requires:** `ultralytics` (YOLOv8), `opencv-python`, `requests`, `pybaseball`.
Steps 2–3 require network access (Savant clip downloads).


## Setup

In [ ]:
import sys, subprocess
from pathlib import Path

REPO = Path().resolve()
while not (REPO / "Figures").exists() and REPO.parent != REPO:
    REPO = REPO.parent

CV    = REPO / "Computer Vision"
PIPE  = CV / "CV Detection Pipeline"
CORE  = CV / "Statcast Analysis Core"
DATA  = REPO / "data"

for p in [str(PIPE), str(CORE)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("Repo root:", REPO)
print("Pipeline: ", PIPE)
print("Core:     ", CORE)


## Configuration

Set the runner and year here — all steps below use these variables.

In [ ]:
# ── Configure your runner ─────────────────────────────────────────────────────
RUNNER_ID   = 647304        # MLB player_id  (647304 = Josh Naylor)
RUNNER_NAME = "naylor"      # lowercase tag — used in filenames
YEAR        = 2025

OUT_DIR  = CV / "archive" / "Archived Runner Runs" / f"{RUNNER_NAME.capitalize()}_{YEAR}"
LEADS    = OUT_DIR / f"{RUNNER_NAME}{YEAR}_leads.csv"
TARGETS  = OUT_DIR / f"{RUNNER_NAME}{YEAR}_targets.csv"
RESULTS  = OUT_DIR / "pilot_results.csv"

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Runner: {RUNNER_NAME} ({RUNNER_ID})  Year: {YEAR}")
print(f"Output: {OUT_DIR}")


## Step 1 — Fetch Per-Attempt Leads from Savant

In [ ]:
# fetch_leads.py — pulls gain_to_release_ft per attempt from Baseball Savant
# Source:
#!/usr/bin/env python3
"""
fetch_leads.py  —  Per-attempt Statcast base-stealing leads for any runner/year
================================================================================

Pulls each tracked steal attempt's lead distances straight from Baseball Savant's
basestealing-running-game service (the data behind the "All Stolen Base Attempts"
drawer on https://baseballsavant.mlb.com/leaderboard/basestealing-run-value):

    GET /leaderboard/services/basestealing-running-game/{runner_id}
        ?season_start={Y}&season_end={Y}
    -> {"data": [ {per-attempt row}, ... ]}

Field mapping (verified against the public leaderboard drawer):
    r_primary_lead         -> lead_at_firstmove_ft   (lead at pitcher's first move)
    r_sec_minus_prim_lead  -> gain_to_release_ft     (lead distance gained)
    r_secondary_lead       -> lead_at_release_ft     (lead at pitch release)
    runs_stolen_on_running_act -> run_value
    runner_moved_cd        -> result (SB/CS)
    target_base            -> base (2B/3B)
    play_id                -> Film-Room GUID (feeds fetch_clips.py directly)

Also (optional) resolves each attempt's game_pk from the runner's StatsAPI gameLog
(date -> gamePk) and writes a targets CSV for fetch_clips.py --targets.

Usage
-----
python3 cv_pilot/fetch_leads.py 647304 2025 \
    --runner-name naylor \
    --out cv_pilot/Naylor_2025/naylor2025_leads.csv \
    --targets-out cv_pilot/Naylor_2025/naylor2025_targets.csv

Host is baseballsavant.mlb.com (NO hyphen). The shell sandbox has no DNS for it,
so run with dangerouslyDisableSandbox.
"""
from __future__ import annotations
import argparse
import csv
import sys
import time
from pathlib import Path

try:
    import requests
except ImportError:
    sys.exit("requests not installed.  pip install requests")

UA = ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
      "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36")
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": UA})

LEADS_URL = ("https://baseballsavant.mlb.com/leaderboard/services/"
             "basestealing-running-game/{rid}?season_start={y}&season_end={y}")
GAMELOG_URL = ("https://statsapi.mlb.com/api/v1/people/{rid}/stats"
               "?stats=gameLog&group=hitting&season={y}&gameType=R")

LEADS_COLS = ["date", "play_id", "pitcher_id", "pitcher_name",
              "catcher_id", "catcher_name", "fielder_name",
              "base", "result", "run_value",
              "lead_at_firstmove_ft", "gain_to_release_ft", "lead_at_release_ft"]


def _get_json(url, tries=4):
    for i in range(tries):
        try:
            r = SESSION.get(url, timeout=30)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if i == tries - 1:
                print(f"  ! GET failed {url}: {e}")
                return None
            time.sleep(1.0 + i)
    return None


def _f(x, nd=None):
    try:
        v = float(x)
        return round(v, nd) if nd is not None else v
    except (TypeError, ValueError):
        return None


def fetch_leads(runner_id: int, year: int) -> list[dict]:
    d = _get_json(LEADS_URL.format(rid=runner_id, y=year))
    data = d.get("data", []) if isinstance(d, dict) else (d or [])
    rows = []
    for a in data:
        rows.append({
            "date": str(a.get("game_date", ""))[:10],
            "play_id": a.get("play_id"),
            "pitcher_id": a.get("pitcher_id"),
            "pitcher_name": a.get("pitcher_name"),
            "catcher_id": a.get("catcher_id"),
            "catcher_name": a.get("catcher_name"),
            "fielder_name": a.get("fielder_name"),
            "base": a.get("target_base"),
            "result": a.get("runner_moved_cd"),
            "run_value": _f(a.get("runs_stolen_on_running_act"), 3),
            "lead_at_firstmove_ft": _f(a.get("r_primary_lead"), 1),
            "gain_to_release_ft": _f(a.get("r_sec_minus_prim_lead"), 1),
            "lead_at_release_ft": _f(a.get("r_secondary_lead"), 1),
        })
    rows.sort(key=lambda r: (r["date"] or "", str(r["pitcher_name"])))
    return rows


def date_to_gamepks(runner_id: int, year: int) -> dict[str, list[int]]:
    d = _get_json(GAMELOG_URL.format(rid=runner_id, y=year))
    out: dict[str, list[int]] = {}
    if not d or not d.get("stats"):
        return out
    for s in d["stats"][0]["splits"]:
        dt = s.get("date")
        pk = s.get("game", {}).get("gamePk")
        if dt and pk:
            out.setdefault(dt, []).append(int(pk))
    return out


def statsapi_sb_cs(runner_id: int, year: int) -> tuple[int, int]:
    d = _get_json(GAMELOG_URL.format(rid=runner_id, y=year))
    sb = cs = 0
    if d and d.get("stats"):
        for s in d["stats"][0]["splits"]:
            st = s["stat"]
            sb += int(st.get("stolenBases", 0))
            cs += int(st.get("caughtStealing", 0))
    return sb, cs


def main():
    ap = argparse.ArgumentParser(description="Fetch per-attempt steal leads for a runner/year")
    ap.add_argument("runner_id", type=int)
    ap.add_argument("year", type=int)
    ap.add_argument("--runner-name", default="runner", help="lowercase tag for clip naming")
    ap.add_argument("--out", required=True, help="leads CSV path")
    ap.add_argument("--targets-out", help="also write a fetch_clips targets CSV (resolves game_pk)")
    args = ap.parse_args()

    rows = fetch_leads(args.runner_id, args.year)
    if not rows:
        sys.exit("No attempts returned by the leads endpoint.")

    out = Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=LEADS_COLS)
        w.writeheader()
        w.writerows(rows)
    n_sb = sum(1 for r in rows if r["result"] == "SB")
    n_cs = sum(1 for r in rows if r["result"] == "CS")
    print(f"[write] {out}  ({len(rows)} attempts: {n_sb} SB / {n_cs} CS)")

    # sanity-check coverage vs StatsAPI season totals
    sb, cs = statsapi_sb_cs(args.runner_id, args.year)
    print(f"[check] StatsAPI {args.year} totals: {sb} SB / {cs} CS  "
          f"(tracked here: {n_sb}/{n_cs}; untracked = steals of home / non-2B-3B / unmeasured)")

    if args.targets_out:
        d2g = date_to_gamepks(args.runner_id, args.year)
        trows = []
        unresolved = 0
        for r in rows:
            pks = d2g.get(r["date"], [])
            if not pks:
                unresolved += 1
            for pk in (pks or [None]):
                trows.append({
                    "game_pk": pk if pk is not None else "",
                    "play_id": r["play_id"],
                    "is_naylor": 1 if args.runner_name.lower() == "naylor" else 0,
                    "clip_prefix": args.runner_name,
                    "date": r["date"],
                    "pitcher_name": r["pitcher_name"],
                    "result": r["result"],
                })
        tout = Path(args.targets_out)
        with open(tout, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=["game_pk", "play_id", "is_naylor",
                                              "clip_prefix", "date", "pitcher_name", "result"])
            w.writeheader()
            w.writerows(trows)
        print(f"[write] {tout}  ({len(trows)} target rows; {unresolved} dates without a gamePk)")


if __name__ == "__main__":
    main()



In [ ]:
sys.path.insert(0, str(CORE))   # fetch_leads lives in Statcast Analysis Core
from fetch_leads import fetch_leads
import pandas as pd

rows = fetch_leads(RUNNER_ID, YEAR, runner_name=RUNNER_NAME,
                   out=str(LEADS), targets_out=str(TARGETS))
df_leads = pd.DataFrame(rows)
print(f"Fetched {len(df_leads)} attempts for {RUNNER_NAME} {YEAR}")
df_leads.head()


## Step 2 — Download Broadcast Clips from Savant

In [ ]:
# fetch_clips.py — downloads mp4 clips via Baseball Savant's Film Room
# Source:
#!/usr/bin/env python3
"""
fetch_clips.py  —  Find & download Statcast pitch clips from Baseball Savant
============================================================================

Discovers pitch video clips on Savant *itself* and downloads them with full
relational metadata, so the CV pilot's outputs join straight back to the
Naylor Model (game_pk + at_bat_number + pitch_number + pitcher_id + catcher_id).

The bridge nobody documents: Savant's game feed
    https://baseballsavant.mlb.com/gf?game_pk=<PK>
returns every pitch with its **play_id** (Film Room GUID) plus pitcher,
catcher, batter, p_throws, ab_number, pitch_number, inning, des, start_speed.
Given a play_id, the Film Room page
    https://baseballsavant.mlb.com/sporty-videos?playId=<GUID>
embeds the downloadable mp4 (https://sporty-clips.mlb.com/....mp4).

So: game_pk → gf feed → (play_id, pitcher, catcher, …) → sporty-videos → mp4.

Usage
-----
# 1) Whole games (all pitches), download the clips:
python3 cv_pilot/fetch_clips.py --game-pks 746161 746158 746150 --download

# 2) Only specific pitches (e.g. the model's SB attempts):
#    attempts.csv must have columns: game_pk,at_bat_number,pitch_number
python3 cv_pilot/fetch_clips.py --attempts attempts.csv --download

# 3) Reverse-resolve known play_ids to full metadata (scan a team's games):
python3 cv_pilot/fetch_clips.py --resolve PLAYID1,PLAYID2 \
        --season 2024 --home-team 119 --opp 135,120,113 --download

# Manifest only (no video download):
python3 cv_pilot/fetch_clips.py --game-pks 746161

Outputs
-------
cv_pilot/clips_manifest.csv   one row per resolved pitch (relational keys + mp4_url + clip_id)
cv_pilot/clips_meta.csv       the subset the detector reads (clip_id, pitcher/catcher, join keys)
cv_pilot/clips/<clip_id>.mp4  downloaded video (with --download; gitignored)
"""

from __future__ import annotations
import argparse
import html
import re
import sys
import time
import unicodedata
from pathlib import Path

import pandas as pd

try:
    import requests
except ImportError:
    sys.exit("requests not installed.  pip install requests")

ROOT          = Path(__file__).resolve().parent
CLIPS_DIR     = ROOT / "clips"
MANIFEST_CSV  = ROOT / "clips_manifest.csv"
META_CSV      = ROOT / "clips_meta.csv"

UA = ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
      "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36")
GF_URL     = "https://baseballsavant.mlb.com/gf?game_pk={pk}"
SPORTY_URL = "https://baseballsavant.mlb.com/sporty-videos?playId={pid}"
SCHED_URL  = ("https://statsapi.mlb.com/api/v1/schedule?sportId=1&season={season}"
              "&teamId={team}&opponentId={opp}&gameType=R")
MP4_RE = re.compile(r"https?://sporty-clips\.mlb\.com/[^\"' ]+\.mp4", re.I)

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": UA})
SLEEP = 0.5      # be polite between requests

# Per-pitch fields we keep (all are relational / descriptive)
GF_FIELDS = ["play_id", "pitcher", "pitcher_name", "catcher", "catcher_name",
             "batter", "batter_name", "p_throws", "ab_number", "pitch_number",
             "inning", "des", "start_speed", "pitch_type", "call_name"]


def _get_json(url, tries=3):
    for i in range(tries):
        try:
            r = SESSION.get(url, timeout=30)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if i == tries - 1:
                print(f"  ! GET failed {url}: {e}")
                return None
            time.sleep(1.0 + i)
    return None


def gf_records(game_pk: int) -> list[dict]:
    """All pitches in a game, with relational metadata, from the Savant gf feed."""
    d = _get_json(GF_URL.format(pk=game_pk))
    if not d:
        return []
    out = []
    for side in ("team_home", "team_away"):
        for p in d.get(side, []):
            if not p.get("play_id"):
                continue
            rec = {k: p.get(k) for k in GF_FIELDS}
            rec["game_pk"] = game_pk
            rec["at_bat_number"] = p.get("ab_number")   # gf ab_number == statcast at_bat_number
            out.append(rec)
    time.sleep(SLEEP)
    return out


def sporty_mp4_url(play_id: str) -> str | None:
    """Scrape the Film Room page for the downloadable mp4 URL."""
    try:
        r = SESSION.get(SPORTY_URL.format(pid=play_id), timeout=30)
        r.raise_for_status()
    except Exception as e:
        print(f"  ! sporty page failed {play_id}: {e}")
        return None
    m = MP4_RE.search(r.text)
    time.sleep(SLEEP)
    # the scraped URL can carry HTML entities (e.g. trailing == shows as
    # &#x3D;&#x3D;); unescape so the link actually resolves
    return html.unescape(m.group(0)) if m else None


def _ascii_last(name: str) -> str:
    """'Emilio Pagán' -> 'pagan' (ascii, lowercase last token)."""
    if not isinstance(name, str) or not name.strip():
        return "pitcher"
    last = name.strip().split()[-1]
    norm = unicodedata.normalize("NFKD", last).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z]", "", norm.lower()) or "pitcher"


def clip_id_for(rec: dict) -> str:
    pid8 = str(rec["play_id"])[:8]
    last = _ascii_last(rec.get("pitcher_name"))
    arm = (rec.get("p_throws") or "R").upper()[0]
    return f"{pid8}_{last}_{arm}.mp4"


def download(url: str, dest: Path) -> bool:
    try:
        with SESSION.get(url, stream=True, timeout=60) as r:
            r.raise_for_status()
            dest.parent.mkdir(parents=True, exist_ok=True)
            with open(dest, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 16):
                    f.write(chunk)
        return True
    except Exception as e:
        print(f"  ! download failed {url}: {e}")
        return False


# ─────────────────────────────────────────────────────────────────────────────
# Selection strategies
# ─────────────────────────────────────────────────────────────────────────────
def from_game_pks(game_pks, sb_only=False) -> list[dict]:
    recs = []
    for pk in game_pks:
        print(f"[gf] game_pk={pk}")
        g = gf_records(int(pk))
        if sb_only:
            g = [r for r in g if re.search(r"steal|caught stealing|stole",
                                           str(r.get("des", "")), re.I)]
        recs.extend(g)
    return recs


def from_attempts(csv_path: Path) -> list[dict]:
    """Target exact pitches: attempts.csv with game_pk,at_bat_number,pitch_number."""
    df = pd.read_csv(csv_path)
    need = {"game_pk", "at_bat_number", "pitch_number"}
    if not need.issubset(df.columns):
        sys.exit(f"{csv_path} must have columns {need}")
    want = {(int(r.game_pk), int(r.at_bat_number), int(r.pitch_number))
            for r in df.itertuples()}
    recs = []
    for pk in sorted({k[0] for k in want}):
        print(f"[gf] game_pk={pk}")
        for r in gf_records(pk):
            key = (int(pk), int(r.get("at_bat_number") or -1),
                   int(r.get("pitch_number") or -1))
            if key in want:
                recs.append(r)
    return recs


def from_targets(csv_path: Path) -> list[dict]:
    """Fetch specific plays by play_id (e.g. exact steal pitches from GUMBO).

    targets.csv needs columns: game_pk, play_id  [, is_naylor, clip_prefix]
    Matches each play_id against its game's gf feed to recover full relational
    metadata (at_bat/pitch/catcher/names) and tags the row for clip naming.
    """
    df = pd.read_csv(csv_path)
    need = {"game_pk", "play_id"}
    if not need.issubset(df.columns):
        sys.exit(f"{csv_path} must have columns {need}")
    want = {}
    for r in df.itertuples():
        want[str(r.play_id)] = {
            "is_naylor": int(getattr(r, "is_naylor", 0) or 0),
            "clip_prefix": str(getattr(r, "clip_prefix", "") or ""),
        }
    recs = []
    for pk in sorted(df["game_pk"].astype(int).unique()):
        print(f"[gf] game_pk={pk}")
        for r in gf_records(int(pk)):
            tag = want.get(str(r.get("play_id")))
            if tag is not None:
                r["_is_naylor"] = tag["is_naylor"]
                r["_clip_prefix"] = tag["clip_prefix"]
                recs.append(r)
    found = {str(r["play_id"]) for r in recs}
    missing = set(want) - found
    if missing:
        print(f"  ! play_ids not found in gf feeds: {len(missing)}")
    return recs


def resolve_play_ids(play_ids, season, home_team, opps) -> list[dict]:
    """Reverse-resolve play_ids by scanning a home team's games vs given opponents."""
    targets = set(play_ids)
    pks = []
    for opp in opps:
        d = _get_json(SCHED_URL.format(season=season, team=home_team, opp=opp))
        if not d:
            continue
        for dt in d.get("dates", []):
            for g in dt["games"]:
                if g["teams"]["home"]["team"]["id"] == int(home_team):
                    pks.append(g["gamePk"])
    recs, found = [], set()
    for pk in pks:
        for r in gf_records(pk):
            if r["play_id"] in targets:
                recs.append(r)
                found.add(r["play_id"])
        if found == targets:
            break
    missing = targets - found
    if missing:
        print(f"  ! unresolved play_ids: {missing}")
    return recs


# ─────────────────────────────────────────────────────────────────────────────
def main():
    ap = argparse.ArgumentParser(description="Fetch Savant pitch clips with relational metadata")
    ap.add_argument("--game-pks", nargs="*", type=int, help="game_pk(s) to pull")
    ap.add_argument("--attempts", help="CSV with game_pk,at_bat_number,pitch_number")
    ap.add_argument("--targets", help="CSV with game_pk,play_id[,is_naylor,clip_prefix]")
    ap.add_argument("--out-dir", help="base output dir (clips/, clips_meta.csv, clips_manifest.csv)")
    ap.add_argument("--resolve", help="comma-separated play_ids to reverse-resolve")
    ap.add_argument("--season", type=int, default=2024)
    ap.add_argument("--home-team", type=int, default=119, help="MLBAM team id (default 119 LAD)")
    ap.add_argument("--opp", default="", help="comma-separated opponent team ids for --resolve")
    ap.add_argument("--sb-only", action="store_true",
                    help="with --game-pks, keep only pitches whose des mentions a steal")
    ap.add_argument("--download", action="store_true", help="download the mp4s (else manifest only)")
    args = ap.parse_args()

    global CLIPS_DIR, MANIFEST_CSV, META_CSV
    if args.out_dir:
        base = Path(args.out_dir)
        base.mkdir(parents=True, exist_ok=True)
        CLIPS_DIR    = base / "clips"
        MANIFEST_CSV = base / "clips_manifest.csv"
        META_CSV     = base / "clips_meta.csv"

    if args.targets:
        recs = from_targets(Path(args.targets))
    elif args.attempts:
        recs = from_attempts(Path(args.attempts))
    elif args.resolve:
        opps = [int(x) for x in args.opp.split(",") if x.strip()]
        if not opps:
            sys.exit("--resolve needs --opp <team_ids>")
        recs = resolve_play_ids(args.resolve.split(","), args.season, args.home_team, opps)
    elif args.game_pks:
        recs = from_game_pks(args.game_pks, sb_only=args.sb_only)
    else:
        sys.exit("Provide one of: --game-pks / --attempts / --targets / --resolve")

    if not recs:
        sys.exit("No records resolved.")

    # build manifest + resolve mp4 urls (+ optional download)
    for r in recs:
        cid = clip_id_for(r)
        # name the clip Naylor actually faced so it is self-identifying
        if r.get("_is_naylor"):
            cid = "NAYLOR_" + cid
        elif r.get("_clip_prefix"):
            cid = f"{r['_clip_prefix']}_" + cid
        r["clip_id"] = cid
        r["is_naylor"] = int(r.get("_is_naylor", 0) or 0)
        r["mp4_url"] = sporty_mp4_url(r["play_id"])
        if args.download and r["mp4_url"]:
            dest = CLIPS_DIR / r["clip_id"]
            ok = download(r["mp4_url"], dest)
            r["downloaded"] = ok
            print(f"  {'✓' if ok else '✗'} {r['clip_id']}  ({r.get('pitcher_name')} -> {r.get('catcher_name')})")
        else:
            r["downloaded"] = False

    man = pd.DataFrame(recs)
    man.to_csv(MANIFEST_CSV, index=False)
    print(f"\n[write] {MANIFEST_CSV}  ({len(man)} pitches)")

    # detector-facing metadata (relational join keys + names)
    meta_cols = ["clip_id", "pitcher_name", "pitcher_id", "p_throws",
                 "catcher_name", "catcher_id", "batter_name", "play_id",
                 "game_pk", "at_bat_number", "pitch_number", "is_naylor"]
    meta = man.rename(columns={"pitcher": "pitcher_id", "catcher": "catcher_id"})
    meta = meta.reindex(columns=meta_cols)
    meta.to_csv(META_CSV, index=False)
    print(f"[write] {META_CSV}  (feeds extract_delivery.py)")
    if args.download:
        n = int(man["downloaded"].sum())
        print(f"[done] downloaded {n}/{len(man)} clips to {CLIPS_DIR}/")
    else:
        print("[note] manifest only — re-run with --download to fetch mp4s")


if __name__ == "__main__":
    main()



In [ ]:
# Download clips for the targets we just fetched
# NOTE: requires network access; produces <OUT_DIR>/clips/*.mp4
result = subprocess.run([
    sys.executable, str(PIPE / "fetch_clips.py"),
    "--targets", str(TARGETS),
    "--out-dir", str(OUT_DIR),
    "--download",
], capture_output=True, text=True)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])


## Step 3 — Extract Delivery Times (YOLOv8-pose)

The detector reads each clip and measures:
- **first_movement_time** — when the pitcher initiates from the stretch
- **ball_release_time** — when the ball leaves the hand
- **delivery_s** = `ball_release_time − first_movement_time`

⚠️ This step is compute-intensive (~10–30 s per clip on MPS/CPU).
Pass `--no-qa` to skip the annotated QA videos and run ~2× faster.

In [ ]:
# extract_delivery.py — YOLOv8-pose delivery-time detector (910 lines)
# Source:
#!/usr/bin/env python3
"""
extract_delivery.py  —  Pitch-delivery detector for Statcast base-stealing clips
================================================================================

Reads broadcast mp4 clips of stolen-base attempts and measures, per clip:

        delivery_s = ball_release_time  -  first_movement_time

where first movement = the pitcher initiating the delivery (from the stretch /
slide-step) and ball release = the ball leaving the hand.  Outputs a per-clip
results row plus an annotated QA video so each measurement can be eyeballed.

Pipeline (see cv_pilot plan):
  A  read frames + true container fps
  B  scene-cut split       (enumerate shots; pick the one with the full delivery,
                           since a steal clip may open on a runner-on-first angle)
  C  YOLOv8-pose + pitcher selection/tracking  (crowded MLB frames)
  D  "set" window          (low-motion plateau before delivery)
  E  first movement        (motion-energy onset; hand-break / leg-lift cross-check)
  F  ball release          (peak throwing-hand speed + reach, parabolic sub-frame)
  G  delivery time + confidence flags
  H  QA overlay video

Usage:
    python3 cv_pilot/extract_delivery.py
    python3 cv_pilot/extract_delivery.py --clip 0001_glasnow_R.mp4
    python3 cv_pilot/extract_delivery.py --no-qa     # skip annotated videos (faster)

Input:
    cv_pilot/clips/*.mp4         hand-dropped pilot clips
    cv_pilot/clips_meta.csv      clip_id,pitcher_name,p_throws[,pitcher_id,bbox_hint]
                                 bbox_hint optional: "x0 y0 x1 y1" (early-frame pitcher box)

Output:
    cv_pilot/pilot_results.csv
    cv_pilot/qa/<clip_id>_annotated.mp4
"""

from __future__ import annotations
import argparse
import sys
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import cv2
except ImportError:
    sys.exit("opencv-python not installed.  pip install -r cv_pilot/requirements.txt")

# ─────────────────────────────────────────────────────────────────────────────
# Paths & constants
# ─────────────────────────────────────────────────────────────────────────────
ROOT        = Path(__file__).resolve().parent
CLIPS_DIR   = ROOT / "clips"
QA_DIR      = ROOT / "qa"
META_PATH   = ROOT / "clips_meta.csv"
RESULTS_CSV = ROOT / "pilot_results.csv"
YOLO_WEIGHTS = "yolov8m-pose.pt"        # downloaded on first run

# Inference knobs (set from CLI in main()).  DEVICE=None lets ultralytics
# auto-select; on Apple-Silicon we resolve it to "mps" so the M2 GPU is used
# instead of CPU (the single biggest speedup).  IMGSZ trades detail for speed.
DEVICE = None
IMGSZ  = 640
HALF   = False
# Optional cap (seconds) on how much of each clip to analyse.  Steal broadcasts
# put the delivery near the start, then pan to follow the runner; that late
# runner-slide motion can outweigh the pitch and pull the detector off-target.
# Capping trims it — faster AND more robust on continuous follow-shots.
# None = analyse the whole clip (preserves the validated pool behaviour).
MAX_ANALYSIS_S = None

# Speed note (measured, negative result — do not re-attempt the two-pass idea):
# inference is ~90 ms/frame on MPS and batching does NOT help (the forward pass
# saturates 78-96 ms at any batch size), so the only accuracy-safe lever is
# running FEWER frames.  A two-pass "coarse-localize then densely detect only the
# window" prototype was A/B'd on 12 labelled clips: it was NOT faster (389.95s ->
# 396.33s) and MAE was neutral (usable delivery_s 0.147 -> 0.148s).  Reason: the
# per-segment scan in process_clip already SHORT-CIRCUITS on the first clean,
# in-band, well-tracked delivery (see the `break` below), so it implicitly stops
# at the live pitch already — a coarse pre-pass is pure added overhead, and a
# crude motion-spread localizer also mis-locks onto later replay segments.  If a
# real speedup is ever needed, the lever is fewer frames via MAX_ANALYSIS_S
# (window-cap), not a second pass.

# First-move detection method (set from CLI):
#   "heel" — DEFAULT.  Refine the lead-leg displacement crossing FORWARD to the
#            lead-ankle's sustained vertical (upward) rise onset — the
#            implementable proxy for the user's manual definition "first move =
#            lead heel lifts to ~10-15deg".  COCO-17 pose has no heel/toe
#            keypoint, so the ankle's vertical lift is the closest signal; it
#            never fires earlier than "disp" (slide-steps fall back).  Validated
#            on 50 hand-labels: usable delivery_s MAE 0.164 -> 0.108s (-30%),
#            bias +0.151 -> +0.053s, vs the "disp" original.
#   "disp" — original first lead-leg total-displacement crossing (reversible).
FIRST_MOVE_METHOD = "heel"

# COCO-17 keypoint indices (YOLOv8-pose)
NOSE = 0
L_SHO, R_SHO = 5, 6
L_ELB, R_ELB = 7, 8
L_WRI, R_WRI = 9, 10
L_HIP, R_HIP = 11, 12
L_KNE, R_KNE = 13, 14
L_ANK, R_ANK = 15, 16

KPT_CONF_MIN   = 0.30      # ignore keypoints below this confidence
SCENE_CUT_CORR = 0.55      # histogram-correlation below this = scene cut
PLAUSIBLE_BAND = (0.80, 1.80)   # delivery_s sanity band (first-move -> release)
MIN_SEG_FRAMES = 40        # a contiguous shot shorter than this can't hold a full delivery

# Pose-skeleton edges for the QA overlay
SKELETON = [(5,7),(7,9),(6,8),(8,10),(5,6),(5,11),(6,12),(11,12),
            (11,13),(13,15),(12,14),(14,16),(0,5),(0,6)]


# ─────────────────────────────────────────────────────────────────────────────
# Small helpers
# ─────────────────────────────────────────────────────────────────────────────
def throwing_wrist(p_throws: str) -> int:
    return R_WRI if str(p_throws).upper().startswith("R") else L_WRI

def throwing_elbow(p_throws: str) -> int:
    return R_ELB if str(p_throws).upper().startswith("R") else L_ELB

def lead_ankle(p_throws: str) -> int:
    # RHP strides with the LEFT leg (lead), LHP with the RIGHT leg.
    return L_ANK if str(p_throws).upper().startswith("R") else R_ANK


def iou(b1, b2) -> float:
    ax0, ay0, ax1, ay1 = b1
    bx0, by0, bx1, by1 = b2
    ix0, iy0 = max(ax0, bx0), max(ay0, by0)
    ix1, iy1 = min(ax1, bx1), min(ay1, by1)
    iw, ih = max(0.0, ix1 - ix0), max(0.0, iy1 - iy0)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    a1 = (ax1 - ax0) * (ay1 - ay0)
    a2 = (bx1 - bx0) * (by1 - by0)
    return inter / (a1 + a2 - inter + 1e-9)


def parabolic_peak(y_prev: float, y0: float, y_next: float) -> float:
    """Sub-frame offset (in [-0.5, 0.5]) of a peak given 3 samples around it."""
    denom = (y_prev - 2.0 * y0 + y_next)
    if abs(denom) < 1e-9:
        return 0.0
    off = 0.5 * (y_prev - y_next) / denom
    return float(np.clip(off, -0.5, 0.5))


def interp_crossing(idx_below: int, v_below: float, v_above: float, thresh: float) -> float:
    """Linear sub-frame frame index where a rising signal crosses `thresh`."""
    if v_above == v_below:
        return float(idx_below)
    frac = (thresh - v_below) / (v_above - v_below)
    return idx_below + float(np.clip(frac, 0.0, 1.0))


# ─────────────────────────────────────────────────────────────────────────────
# B — scene-cut guard
# ─────────────────────────────────────────────────────────────────────────────
def first_scene_cut(frames: list[np.ndarray]) -> int:
    """Return the frame index of the first scene cut, or len(frames) if none."""
    prev_hist = None
    for i, f in enumerate(frames):
        hsv = cv2.cvtColor(f, cv2.COLOR_BGR2HSV)
        hist = cv2.calcHist([hsv], [0, 1], None, [50, 60], [0, 180, 0, 256])
        cv2.normalize(hist, hist, 0, 1, cv2.NORM_MINMAX)
        if prev_hist is not None:
            corr = cv2.compareHist(prev_hist, hist, cv2.HISTCMP_CORREL)
            if corr < SCENE_CUT_CORR and i > 3:   # ignore the very first frames
                return i
        prev_hist = hist
    return len(frames)


def scene_segments(frames: list[np.ndarray]) -> list[tuple[int, int]]:
    """Split the clip into contiguous shots at EVERY scene cut.

    Broadcast steal clips sometimes OPEN on a runner-on-first / different
    camera angle and only show the pitcher's full throw in a later shot.  So we
    can't assume the delivery is in the opening segment — we enumerate all shots
    and let the caller pick the one where the pitcher's throw is fully visible.

    Returns a list of (start, end) frame-index pairs (end exclusive).
    """
    prev_hist = None
    cuts = [0]
    for i, f in enumerate(frames):
        hsv = cv2.cvtColor(f, cv2.COLOR_BGR2HSV)
        hist = cv2.calcHist([hsv], [0, 1], None, [50, 60], [0, 180, 0, 256])
        cv2.normalize(hist, hist, 0, 1, cv2.NORM_MINMAX)
        if prev_hist is not None:
            corr = cv2.compareHist(prev_hist, hist, cv2.HISTCMP_CORREL)
            # require a gap from the previous cut so a multi-frame dissolve
            # doesn't register as several cuts in a row
            if corr < SCENE_CUT_CORR and (i - cuts[-1]) > 3:
                cuts.append(i)
        prev_hist = hist
    cuts.append(len(frames))
    return [(cuts[k], cuts[k + 1]) for k in range(len(cuts) - 1)]


# ─────────────────────────────────────────────────────────────────────────────
# C — YOLOv8-pose detection + pitcher selection / tracking
# ─────────────────────────────────────────────────────────────────────────────
def run_pose(frames: list[np.ndarray], model):
    """Per-frame multi-person pose with ByteTrack IDs.

    Returns dict: track_id -> list of (frame_idx, bbox, kpts[17,3]).
    kpts columns = (x, y, conf).
    """
    tracks: dict[int, list] = {}
    for fi, frame in enumerate(frames):
        res = model.track(frame, persist=True, verbose=False,
                          classes=[0], tracker="bytetrack.yaml",
                          device=DEVICE, imgsz=IMGSZ, half=HALF)[0]
        if res.boxes is None or res.boxes.id is None or res.keypoints is None:
            continue
        ids   = res.boxes.id.cpu().numpy().astype(int)
        boxes = res.boxes.xyxy.cpu().numpy()
        kxy   = res.keypoints.xy.cpu().numpy()              # (n,17,2)
        kcf   = res.keypoints.conf
        kcf   = (kcf.cpu().numpy() if kcf is not None
                 else np.ones(kxy.shape[:2], dtype=float))
        for tid, box, xy, cf in zip(ids, boxes, kxy, kcf):
            kpts = np.concatenate([xy, cf[:, None]], axis=1)  # (17,3)
            tracks.setdefault(int(tid), []).append((fi, box, kpts))
    return tracks


def track_motion_energy(seq, diag_ref: float) -> float:
    """Total scale-normalised keypoint motion across a track's frames."""
    total = 0.0
    for (_, _, k0), (_, _, k1) in zip(seq[:-1], seq[1:]):
        good = (k0[:, 2] > KPT_CONF_MIN) & (k1[:, 2] > KPT_CONF_MIN)
        if not good.any():
            continue
        d = np.linalg.norm(k1[good, :2] - k0[good, :2], axis=1)
        total += float(d.mean())
    return total / max(diag_ref, 1.0)


def select_pitcher(tracks, frame_w, frame_h, bbox_hint=None):
    """Pick the pitcher track by center-x proximity + delivery motion energy.

    If bbox_hint is given, pick the track whose early bbox best matches it.
    Returns (track_id, seq) or (None, None).
    """
    if not tracks:
        return None, None
    diag = float(np.hypot(frame_w, frame_h))

    if bbox_hint is not None:
        best, best_iou = None, 0.0
        for tid, seq in tracks.items():
            early = [b for (_, b, _) in seq[:8]]
            if not early:
                continue
            m = max(iou(b, bbox_hint) for b in early)
            if m > best_iou:
                best, best_iou = tid, m
        if best is not None and best_iou > 0.1:
            return best, tracks[best]

    cx = frame_w / 2.0
    best, best_score = None, -1e18
    for tid, seq in tracks.items():
        if len(seq) < 4:
            continue
        boxes = np.array([b for (_, b, _) in seq])
        bcx = ((boxes[:, 0] + boxes[:, 2]) / 2.0).mean()
        center_prox = 1.0 - abs(bcx - cx) / (frame_w / 2.0)   # 1=center, 0=edge
        energy = track_motion_energy(seq, diag)
        score = 1.0 * center_prox + 2.5 * energy              # energy dominates
        if score > best_score:
            best, best_score = tid, score
    return best, (tracks[best] if best is not None else None)


# ─────────────────────────────────────────────────────────────────────────────
# Build dense per-frame keypoint arrays for the chosen pitcher
# ─────────────────────────────────────────────────────────────────────────────
def densify(seq, n_frames):
    """seq -> (kpts[n_frames,17,3], boxes[n_frames,4]) with NaN gaps filled."""
    kpts  = np.full((n_frames, 17, 3), np.nan)
    boxes = np.full((n_frames, 4), np.nan)
    for (fi, box, k) in seq:
        kpts[fi]  = k
        boxes[fi] = box
    # forward/back-fill bbox for overlay continuity
    for c in range(4):
        s = pd.Series(boxes[:, c]).ffill().bfill()
        boxes[:, c] = s.values
    return kpts, boxes


def smooth(x, win=3):
    """Light centered moving average ignoring NaN."""
    s = pd.Series(x)
    return s.rolling(win, center=True, min_periods=1).mean().values


# ─────────────────────────────────────────────────────────────────────────────
# D/E/F — set window, first movement, release
# ─────────────────────────────────────────────────────────────────────────────
def motion_energy_series(kpts, diag_ref):
    """Per-frame scale-normalised mean keypoint displacement."""
    n = len(kpts)
    me = np.zeros(n)
    for i in range(1, n):
        k0, k1 = kpts[i - 1], kpts[i]
        good = (k0[:, 2] > KPT_CONF_MIN) & (k1[:, 2] > KPT_CONF_MIN)
        if not good.any():
            me[i] = np.nan
            continue
        d = np.linalg.norm(k1[good, :2] - k0[good, :2], axis=1)
        me[i] = d.mean() / max(diag_ref, 1.0)
    me = pd.Series(me).interpolate().bfill().ffill().values
    return smooth(me, 3)


def find_set_and_first_move(me):
    """Return (set_end_idx, first_move_subframe, floor, std)."""
    n = len(me)
    # Peak motion = somewhere in the delivery; search the set BEFORE it.
    peak = int(np.nanargmax(me))
    search_hi = max(peak, 5)
    # Rolling std to find the quietest plateau before the peak
    win = 5
    best_i, best_var = 0, 1e18
    for i in range(0, max(1, search_hi - win)):
        seg = me[i:i + win]
        v = float(np.nanstd(seg))
        if v < best_var:
            best_var, best_i = v, i
    set_lo = best_i
    set_hi = min(best_i + win, search_hi)
    floor = float(np.nanmean(me[set_lo:set_hi]))
    std   = float(np.nanstd(me[set_lo:set_hi])) + 1e-6
    thresh = floor + 4.0 * std

    # First sustained crossing after the set plateau
    fm = None
    for i in range(set_hi, peak + 1):
        if me[i] > thresh and np.all(me[i:min(i + 3, n)] > floor + 2.0 * std):
            fm = interp_crossing(i - 1, me[i - 1], me[i], thresh)
            break
    if fm is None:
        fm = float(set_hi)
    return set_hi, fm, floor, std


def leg_lift_frame(kpts, p_throws, set_end, peak, diag_ref):
    """PRIMARY first-move signal: the lead leg leaves its planted stance.

    Per the project (and user) definition, "first movement" = the lead leg
    starting to move — *upwards OR horizontally* (slide-steps barely lift the
    foot but always shift it).  The old version watched only the lead ankle's
    vertical rise against a hard pixel floor, which missed subtle / horizontal
    loading and fired tens of frames late.

    Fix: track the **total displacement** (√Δx²+Δy²) of BOTH the lead ankle and
    lead knee from a *rolling* baseline (median of the recent planted position,
    so a slow camera pan doesn't count as motion).  First move = the first frame
    after the set where that displacement clears a noise-relative threshold and
    stays elevated.  Scale-normalised by the frame diagonal so the threshold is
    framing-independent.  Returns a sub-frame float index, or None if both
    lead-leg keypoints are too occluded.
    """
    la = lead_ankle(p_throws)
    lk = L_KNE if str(p_throws).upper().startswith("R") else R_KNE
    if (np.isfinite(np.where(kpts[:, la, 2] > KPT_CONF_MIN, kpts[:, la, 1], np.nan)).sum() < 5
            and np.isfinite(np.where(kpts[:, lk, 2] > KPT_CONF_MIN, kpts[:, lk, 1], np.nan)).sum() < 5):
        return None

    def pos(idx):
        x = np.where(kpts[:, idx, 2] > KPT_CONF_MIN, kpts[:, idx, 0], np.nan)
        y = np.where(kpts[:, idx, 2] > KPT_CONF_MIN, kpts[:, idx, 1], np.nan)
        x = pd.Series(x).interpolate().bfill().ffill().values
        y = pd.Series(y).interpolate().bfill().ffill().values
        return x, y

    ax, ay = pos(la)
    kx, ky = pos(lk)
    n = len(ax)

    # FIXED baseline = the planted lead-leg position during the set.  We anchor
    # it on the *quietest* sub-window of the pre-set frames, NOT the raw window
    # edge — otherwise frames where the pitcher is still settling INTO the set
    # contaminate both the baseline position and the noise estimate (observed:
    # a 16 px threshold that hid the real first move).  (A rolling baseline is
    # also wrong here: it tracks an oscillating keypoint and hides the move.)
    w0 = max(0, set_end - 18)
    w1 = max(w0 + 7, set_end + 2)

    def disp_from(px, py, lo, hi):
        bx = float(np.nanmedian(px[lo:hi]))
        by = float(np.nanmedian(py[lo:hi]))
        return np.hypot(px - bx, py - by)

    # pass 1: rough displacement to locate the quietest 7-frame plateau
    d0 = np.maximum(disp_from(ax, ay, w0, w1), disp_from(kx, ky, w0, w1))
    qs, qbest = w0, 1e18
    for j in range(w0, max(w0 + 1, w1 - 7)):
        v = float(np.nanstd(d0[j:j + 7]))
        if v < qbest:
            qbest, qs = v, j
    qe = qs + 7

    # pass 2: planted baseline + noise from that quiet plateau
    m = np.maximum(disp_from(ax, ay, qs, qe), disp_from(kx, ky, qs, qe)) / max(diag_ref, 1.0)
    m = smooth(m, 3)
    noise = float(np.nanstd(m[qs:qe]))
    # floor ≈ 7 px on a 1280×720 clip: clears pre-pitch micro-drift / camera sway
    # so we fire on the genuine delivery initiation, not the settle.
    thresh = max(4.0 * noise, 0.0048)

    start = max(1, set_end - 3)
    hi = max(start + 1, peak + 1)
    disp_cross = None
    for i in range(start, min(hi, n)):
        # commit on first crossing whose *median* over the next 5 frames stays
        # elevated — median ignores 1–2 flicker dropouts (robust to noisy joints)
        if m[i] > thresh and float(np.nanmedian(m[i:min(i + 5, n)])) > thresh:
            disp_cross = (i, interp_crossing(i - 1, m[i - 1], m[i], thresh))
            break
    if disp_cross is None:
        return None
    i0, cross = disp_cross

    # ── heel-lift refinement (user's manual definition) ──────────────────────
    # The displacement crossing fires on the FIRST lead-leg motion, which our
    # 50 hand-labels show lands ~4 frames (~0.06s) *before* the user's mark of
    # "heel lifts to ~10-15deg".  When method="heel", advance the trigger to the
    # lead ankle's sustained UPWARD rise (heel coming off the ground): y in image
    # coords decreases as the foot lifts.  Pure horizontal slide-steps never
    # clear the vertical floor, so they fall back to the displacement crossing
    # (this refinement can only DELAY the trigger, never advance it).
    if FIRST_MOVE_METHOD == "heel":
        by = float(np.nanmedian(ay[qs:qe]))            # planted lead-ankle height
        up = (by - ay) / max(diag_ref, 1.0)            # >0 when ankle rises
        up = smooth(up, 3)
        nv = float(np.nanstd(up[qs:qe]))
        vthresh = max(3.5 * nv, 0.010)                 # ≈ heel clearly off the rubber
        for i in range(i0, min(hi, n)):
            if up[i] > vthresh and float(np.nanmedian(up[i:min(i + 5, n)])) > vthresh:
                return interp_crossing(i - 1, up[i - 1], up[i], vthresh)
        # never cleared the vertical floor (slide-step / occluded) → keep disp
        return cross
    return cross


def hand_break_frame(kpts, set_end, peak):
    """Cross-check: frame where the two wrists start separating."""
    lw, rw = kpts[:, L_WRI, :2], kpts[:, R_WRI, :2]
    cf = np.minimum(kpts[:, L_WRI, 2], kpts[:, R_WRI, 2])
    sep = np.linalg.norm(lw - rw, axis=1)
    sep[cf < KPT_CONF_MIN] = np.nan
    sep = pd.Series(sep).interpolate().bfill().ffill().values
    base = np.nanmean(sep[max(0, set_end - 4):set_end + 1]) + 1e-6
    for i in range(set_end, max(set_end + 1, peak)):
        if sep[i] > 1.5 * base:
            return float(i)
    return None


def find_release(kpts, p_throws, first_move_idx, diag_ref):
    """Release ≈ peak throwing-hand speed × reach, parabolic sub-frame refined."""
    w = throwing_wrist(p_throws)
    sho = R_SHO if w == R_WRI else L_SHO
    hip = R_HIP if w == R_WRI else L_HIP

    wx = pd.Series(np.where(kpts[:, w, 2] > KPT_CONF_MIN, kpts[:, w, 0], np.nan))
    wy = pd.Series(np.where(kpts[:, w, 2] > KPT_CONF_MIN, kpts[:, w, 1], np.nan))
    wx = wx.interpolate().bfill().ffill().values
    wy = wy.interpolate().bfill().ffill().values

    # hand speed (per-frame displacement), scale-normalised
    speed = np.zeros(len(wx))
    speed[1:] = np.hypot(np.diff(wx), np.diff(wy)) / max(diag_ref, 1.0)
    speed = smooth(speed, 3)

    # reach: wrist distance from shoulder-hip body centroid
    # (frames where both shoulder & hip are missing yield NaN; interpolate fills them —
    #  suppress the benign "Mean of empty slice" warning for those all-NaN columns)
    with np.errstate(invalid="ignore"):
        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            bx = np.nanmean([kpts[:, sho, 0], kpts[:, hip, 0]], axis=0)
            by = np.nanmean([kpts[:, sho, 1], kpts[:, hip, 1]], axis=0)
    bx = pd.Series(bx).interpolate().bfill().ffill().values
    by = pd.Series(by).interpolate().bfill().ffill().values
    reach = np.hypot(wx - bx, wy - by) / max(diag_ref, 1.0)
    reach = smooth(reach, 3)

    n = len(speed)
    lo = int(np.ceil(first_move_idx)) + 3
    if lo >= n - 2:
        return float(n - 1), 0.0
    hi = min(n - 1, lo + 95)                 # cap the delivery window (~1.6 s @60fps);
                                             # excludes the far follow-through whip that
                                             # otherwise wins a naive global speed-argmax.

    # Is the throwing wrist actually tracked well? (reach must have real dynamic
    # range — a flat reach means the keypoint is barely moving / poorly detected.)
    rw = reach[lo:hi]
    spread = float(np.nanpercentile(rw, 90) - np.nanpercentile(rw, 10))

    if spread >= 0.010:
        method = "extension_collapse"
        # ── good tracking: extension-then-collapse model ──────────────────────
        # At release the throwing hand is at max forward extension while HIGH
        # (out front), then `reach` collapses as the arm whips down across the
        # body.  Earlier reach humps (arm swinging *down* during the leg lift)
        # are rejected by requiring the wrist to be in the upper part of its
        # vertical travel.  Release = peak hand speed between that high-hand
        # extension peak and the reach-collapse onset.
        wyw = wy[lo:hi]
        wy_gate = float(np.nanmin(wyw) + 0.55 * (np.nanmax(wyw) - np.nanmin(wyw)))
        rg = reach.copy()
        rg[wy > wy_gate] = -1e9              # drop low-hand (arm-swing-down) frames
        rg[:lo] = -1e9
        rg[hi:] = -1e9
        r_ext = int(np.nanargmax(rg))
        if rg[r_ext] <= -1e8:                # all frames gated out — fall back
            r_ext = lo + int(np.nanargmax(reach[lo:hi]))
        peakv = reach[r_ext]

        coll = None
        for j in range(r_ext + 1, min(r_ext + 12, hi)):
            if reach[j] < 0.5 * peakv:
                coll = j
                break
        end_ref = coll if coll is not None else min(r_ext + 8, hi)

        a = max(lo, r_ext)
        b = max(a + 1, int(np.ceil(end_ref)))
        seg = speed[a:b]
        pk = a + int(np.nanargmax(seg)) if seg.size and not np.all(np.isnan(seg)) else r_ext
    else:
        # ── poor wrist tracking (flat reach): reach is uninformative.  Use the
        #    PEAK WRIST HEIGHT (top of the throwing arc) as the release proxy,
        #    but take the FIRST turning point — not the global min — because a
        #    poorly-tracked wrist jumps around in the follow-through and would
        #    otherwise win the global extreme.  Heavy smoothing kills jitter so
        #    the descent-to-top-then-drop reads as one clean local minimum.
        method = "flat_reach_height"
        hh = min(n - 1, lo + 85)
        wysm = smooth(wy, 7)
        rise_margin = 0.004 * diag_ref               # hand must come back down ≥~6 px
        pk = None
        for i in range(lo + 3, hh - 5):
            # local min that the hand clearly comes back DOWN from (wy rises)
            if (wysm[i] <= wysm[i - 1]
                    and float(np.nanmedian(wysm[i + 1:i + 6])) > wysm[i] + rise_margin
                    and wysm[i] < float(np.nanmedian(wysm[lo:lo + 4]))):
                pk = i
                break
        if pk is None:
            seg = wy[lo:hh]
            pk = lo + int(np.nanargmin(seg)) if seg.size and not np.all(np.isnan(seg)) else lo

    conf = float(kpts[pk, w, 2]) if not np.isnan(kpts[pk, w, 2]) else 0.0
    if method == "flat_reach_height":
        conf *= 0.7                          # weaker estimator → flag lower trust
    # sub-frame parabolic refine on the speed curve
    if 0 < pk < n - 1:
        off = parabolic_peak(speed[pk - 1], speed[pk], speed[pk + 1])
    else:
        off = 0.0
    return pk + off, conf, method


# ─────────────────────────────────────────────────────────────────────────────
# H — QA overlay video
# ─────────────────────────────────────────────────────────────────────────────
def write_qa(frames, fps, kpts, boxes, set_i, fm, rel, delivery_s, out_path):
    h, w = frames[0].shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    vw = cv2.VideoWriter(str(out_path), fourcc, fps, (w, h))
    fm_i, rel_i = int(round(fm)), int(round(rel))
    for i, frame in enumerate(frames):
        f = frame.copy()
        k = kpts[i]
        for a, b in SKELETON:
            if k[a, 2] > KPT_CONF_MIN and k[b, 2] > KPT_CONF_MIN:
                pa = (int(k[a, 0]), int(k[a, 1]))
                pb = (int(k[b, 0]), int(k[b, 1]))
                cv2.line(f, pa, pb, (0, 255, 0), 2)
        for j in range(17):
            if k[j, 2] > KPT_CONF_MIN:
                cv2.circle(f, (int(k[j, 0]), int(k[j, 1])), 3, (0, 200, 255), -1)
        if not np.isnan(boxes[i]).any():
            x0, y0, x1, y1 = boxes[i].astype(int)
            cv2.rectangle(f, (x0, y0), (x1, y1), (255, 180, 0), 1)
        label = ""
        color = (255, 255, 255)
        if i == set_i:   label, color = "SET",         (200, 200, 200)
        if i == fm_i:    label, color = "FIRST MOVE",  (0, 255, 255)
        if i == rel_i:   label, color = "RELEASE",     (0, 0, 255)
        if label:
            cv2.rectangle(f, (0, 0), (w, 40), (0, 0, 0), -1)
            cv2.putText(f, label, (10, 28), cv2.FONT_HERSHEY_SIMPLEX,
                        0.9, color, 2)
        cv2.putText(f, f"f{i}  delivery={delivery_s:.3f}s",
                    (10, h - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                    (255, 255, 255), 2)
        vw.write(f)
    vw.release()


# ─────────────────────────────────────────────────────────────────────────────
# Per-clip driver
# ─────────────────────────────────────────────────────────────────────────────
def parse_hint(s):
    if not isinstance(s, str) or not s.strip():
        return None
    try:
        vals = [float(v) for v in s.replace(",", " ").split()]
        return vals if len(vals) == 4 else None
    except ValueError:
        return None


def _detect_in_segment(seg, p_throws, model, diag, bbox_hint=None):
    """Run pose + event detection on one contiguous shot.

    Returns a dict of segment-relative results (or None if no pitcher track).
    All frame indices are relative to the START of this segment.
    """
    tracks = run_pose(seg, model)
    tid, pseq = select_pitcher(tracks, seg[0].shape[1], seg[0].shape[0], bbox_hint)
    if pseq is None:
        return None

    kpts, boxes = densify(pseq, len(seg))
    me = motion_energy_series(kpts, diag)
    peak = int(np.nanargmax(me))

    set_i, motion_onset, floor, std = find_set_and_first_move(me)
    # FIRST MOVEMENT = lead foot leaves the ground (project definition).
    # Priority: leg-lift (primary) -> hand-break -> generic motion onset.
    ll = leg_lift_frame(kpts, p_throws, set_i, peak, diag)
    hb = hand_break_frame(kpts, set_i, peak)
    if ll is not None:
        fm, fm_cue = float(ll), "leg_lift"
    elif hb is not None:
        fm, fm_cue = float(hb), "hand_break"
    else:
        fm, fm_cue = float(motion_onset), "motion_onset"

    rel, rel_conf, rel_method = find_release(kpts, p_throws, fm, diag)
    return {
        "tid": tid, "kpts": kpts, "boxes": boxes,
        "set_i": set_i, "fm": fm, "rel": rel, "fm_cue": fm_cue,
        "rel_conf": rel_conf, "rel_method": rel_method,
        "ll": ll, "hb": hb, "me_peak": float(me[peak]),
    }


def process_clip(path: Path, p_throws: str, model, bbox_hint=None, make_qa=True):
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frames = []
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        frames.append(fr)
    cap.release()

    n_full = len(frames)
    if MAX_ANALYSIS_S is not None and fps > 0:
        cap_n = int(round(MAX_ANALYSIS_S * fps))
        if 0 < cap_n < len(frames):
            frames = frames[:cap_n]

    row = {"clip_id": path.name, "p_throws": p_throws, "fps": round(fps, 3),
           "n_frames": n_full, "analysis_frames": len(frames), "status": "ok"}
    if len(frames) < 8:
        row["status"] = "too_short"
        return row

    h, w = frames[0].shape[:2]
    diag = float(np.hypot(w, h))

    # Enumerate every contiguous shot.  The pitcher's full delivery may NOT be in
    # the opening shot (broadcast can open on a runner-on-first angle), so we
    # scan shots in order and keep the first one that yields a clean, fully
    # visible delivery (trackable throwing hand + in-band timing).  Shorter
    # shots that can't physically contain a delivery are skipped.
    segments = scene_segments(frames)
    candidates = [(s, e) for (s, e) in segments if (e - s) >= MIN_SEG_FRAMES]
    if not candidates:                       # nothing long enough — use longest shot
        candidates = [max(segments, key=lambda se: se[1] - se[0])]
    row["n_segments"] = len(segments)
    row["scene_cut_at"] = segments[0][1] if len(segments) > 1 else -1

    best = None                              # (quality_key, seg, det)
    chosen = None
    for (s, e) in candidates:
        det = _detect_in_segment(frames[s:e], p_throws, model, diag, bbox_hint)
        if det is None:
            continue
        delivery_s = (det["rel"] - det["fm"]) / fps
        in_band = PLAUSIBLE_BAND[0] <= delivery_s <= PLAUSIBLE_BAND[1]
        good_track = (det["rel_method"] != "flat_reach_height")
        good_conf = det["rel_conf"] >= KPT_CONF_MIN
        # rank: a fully-visible throw (good_track) that is in-band and well
        # tracked wins; ties broken by detection confidence then motion energy.
        qkey = (int(good_track), int(in_band), int(good_conf),
                det["rel_conf"], det["me_peak"])
        if best is None or qkey > best[0]:
            best = (qkey, (s, e), det)
        # short-circuit: a clean in-band, well-tracked delivery is good enough
        if good_track and in_band and good_conf:
            chosen = (s, e, det)
            break

    if best is None:
        row["status"] = "no_pitcher"
        return row
    if chosen is None:
        chosen = (best[1][0], best[1][1], best[2])
    s, e, det = chosen

    kpts, boxes = det["kpts"], det["boxes"]
    set_i, fm, rel = det["set_i"], det["fm"], det["rel"]
    rel_conf, rel_method, fm_cue = det["rel_conf"], det["rel_method"], det["fm_cue"]
    ll, hb = det["ll"], det["hb"]
    delivery_s = (rel - fm) / fps

    row["pitcher_track_id"] = det["tid"]
    row["analysis_frames"] = e - s
    row["chosen_segment"] = f"[{s}:{e}]"
    row["segment_index"] = segments.index((s, e)) if (s, e) in segments else -1
    # report event frames in FULL-CLIP coordinates (so they line up with manual
    # labels, which are numbered against the whole clip)
    off = s
    row.update({
        "set_frame": set_i + off,
        "first_move_frame": round(fm + off, 2),
        "release_frame": round(rel + off, 2),
        "delivery_s": round(delivery_s, 4),
        "release_kpt_conf": round(rel_conf, 3),
        "release_method": rel_method,
        "first_move_cue": fm_cue,
        "leg_lift_frame": round(ll + off, 2) if ll is not None else None,
        "hand_break_frame": round(hb + off, 2) if hb is not None else None,
    })

    # confidence flags
    in_band = PLAUSIBLE_BAND[0] <= delivery_s <= PLAUSIBLE_BAND[1]
    good_conf = rel_conf >= KPT_CONF_MIN
    # a delivery detected within a contiguous shot is, by construction, not cut
    # off mid-throw; pre_cut stays True for any successful in-shot detection.
    pre_cut = True
    # `flat_reach_height` means YOLO never tracked the throwing hand (reach was
    # flat) — the release estimate is unreliable, so the clip is NOT usable for
    # accuracy.  It still counts toward coverage (flag what we can't measure
    # rather than emit a silently-wrong release).
    good_track = (rel_method != "flat_reach_height")
    row["good_track"] = bool(good_track)
    row["in_band"] = bool(in_band)
    row["release_pre_cut"] = bool(pre_cut)
    row["usable"] = bool(in_band and pre_cut and good_conf and good_track)
    row["confidence"] = round(
        (0.4 * good_conf + 0.25 * in_band + 0.15 * pre_cut + 0.2 * good_track), 3)

    if make_qa:
        QA_DIR.mkdir(parents=True, exist_ok=True)
        out = QA_DIR / f"{path.stem}_annotated.mp4"
        try:
            # QA overlay uses segment-relative indices on the chosen shot
            write_qa(frames[s:e], fps, kpts, boxes, set_i, fm, rel, delivery_s, out)
            row["qa_video"] = out.name
        except Exception as e2:               # QA is non-critical
            row["qa_video"] = f"ERR:{e2}"
    return row


# ─────────────────────────────────────────────────────────────────────────────
# main
# ─────────────────────────────────────────────────────────────────────────────
def load_meta():
    if META_PATH.exists():
        return pd.read_csv(META_PATH)
    return pd.DataFrame(columns=["clip_id", "pitcher_name", "p_throws",
                                 "pitcher_id", "bbox_hint"])


def main():
    ap = argparse.ArgumentParser(description="Pitch-delivery CV detector")
    ap.add_argument("--clip", help="process a single clip filename in clips/")
    ap.add_argument("--no-qa", action="store_true", help="skip QA videos")
    ap.add_argument("--weights", default=YOLO_WEIGHTS,
                    help="pose weights; yolov8n-pose.pt is ~5-8x faster than -m")
    ap.add_argument("--device", default="auto",
                    help="auto|mps|cpu|cuda|0  (auto picks MPS on Apple Silicon)")
    ap.add_argument("--imgsz", type=int, default=640,
                    help="inference image size (lower = faster, less detail)")
    ap.add_argument("--half", action="store_true", help="FP16 inference where supported")
    ap.add_argument("--max-analysis-s", type=float, default=None,
                    help="only analyse the first N seconds of each clip "
                         "(faster; trims runner-follow tail on continuous shots)")
    ap.add_argument("--clips-dir", help="override clips/ directory")
    ap.add_argument("--meta", help="override clips_meta.csv path")
    ap.add_argument("--out", help="override pilot_results.csv output path")
    ap.add_argument("--qa-dir", help="override qa/ directory")
    ap.add_argument("--first-move", choices=["disp", "heel"], default="heel",
                    help="first-move cue: 'heel' (default; lead-ankle vertical-"
                         "rise onset ~ user's heel-lift definition) or 'disp' "
                         "(original total-displacement crossing)")
    args = ap.parse_args()

    global CLIPS_DIR, QA_DIR, META_PATH, RESULTS_CSV, DEVICE, IMGSZ, HALF, MAX_ANALYSIS_S
    global FIRST_MOVE_METHOD
    FIRST_MOVE_METHOD = args.first_move
    MAX_ANALYSIS_S = args.max_analysis_s
    # Resolve compute device.  On a fanless M2 the GPU (MPS) is far faster than
    # CPU and avoids pegging all cores; fall back to CPU if MPS is unavailable.
    dev = args.device
    if dev == "auto":
        try:
            import torch
            dev = "mps" if torch.backends.mps.is_available() else "cpu"
        except Exception:
            dev = "cpu"
    DEVICE, IMGSZ, HALF = dev, args.imgsz, args.half
    print(f"[device] {DEVICE}  imgsz={IMGSZ}  half={HALF}")
    if args.clips_dir:
        CLIPS_DIR = Path(args.clips_dir)
    if args.qa_dir:
        QA_DIR = Path(args.qa_dir)
    if args.meta:
        META_PATH = Path(args.meta)
    if args.out:
        RESULTS_CSV = Path(args.out)

    if not CLIPS_DIR.exists() or not any(CLIPS_DIR.glob("*.mp4")):
        sys.exit(f"No clips found in {CLIPS_DIR}.  Drop pilot mp4s there first.")

    try:
        from ultralytics import YOLO
    except ImportError:
        sys.exit("ultralytics not installed.  pip install -r cv_pilot/requirements.txt")

    print(f"[load] YOLOv8-pose weights: {args.weights}")
    model = YOLO(args.weights)

    meta = load_meta()
    meta_map = {r["clip_id"]: r for _, r in meta.iterrows()}

    clips = ([CLIPS_DIR / args.clip] if args.clip
             else sorted(CLIPS_DIR.glob("*.mp4")))

    rows = []
    for path in clips:
        if not path.exists():
            print(f"  ! missing {path.name}")
            continue
        m = meta_map.get(path.name, {})
        # infer p_throws from filename suffix (_R / _L) if meta missing
        p_throws = m.get("p_throws")
        if not isinstance(p_throws, str) or not p_throws:
            stem = path.stem.upper()
            p_throws = "L" if stem.endswith("_L") else "R"
        hint = parse_hint(m.get("bbox_hint")) if m is not None else None
        print(f"[clip] {path.name}  (p_throws={p_throws})")
        try:
            row = process_clip(path, p_throws, model, hint, make_qa=not args.no_qa)
        except Exception as e:
            row = {"clip_id": path.name, "status": f"ERR:{e}"}
        # carry relational metadata (join keys to the Naylor model) from meta
        for col in ("pitcher_name", "pitcher_id", "catcher_name", "catcher_id",
                    "batter_name", "play_id", "game_pk", "at_bat_number",
                    "pitch_number", "is_naylor"):
            if col in m and pd.notna(m[col]):
                row[col] = m[col]
        print(f"        -> {row.get('status')}  "
              f"delivery_s={row.get('delivery_s')}  usable={row.get('usable')}")
        rows.append(row)

    df = pd.DataFrame(rows)
    cols = ["clip_id", "pitcher_name", "pitcher_id", "catcher_name", "catcher_id",
            "play_id", "game_pk", "at_bat_number", "pitch_number",
            "p_throws", "fps", "n_frames", "analysis_frames", "scene_cut_at",
            "n_segments", "segment_index", "chosen_segment",
            "set_frame", "first_move_frame", "release_frame", "delivery_s",
            "first_move_cue", "leg_lift_frame", "hand_break_frame",
            "release_method", "release_kpt_conf", "good_track",
            "in_band", "release_pre_cut", "usable",
            "confidence", "qa_video", "pitcher_track_id", "status"]
    df = df.reindex(columns=[c for c in cols if c in df.columns])
    df.to_csv(RESULTS_CSV, index=False)
    print(f"\n[write] {RESULTS_CSV}  ({len(df)} clips)")
    if "usable" in df.columns:
        print(f"        usable: {int(df['usable'].sum())}/{len(df)}")


if __name__ == "__main__":
    main()



In [ ]:
clips_dir = OUT_DIR / "clips"
result = subprocess.run([
    sys.executable, str(PIPE / "extract_delivery.py"),
    "--clips-dir", str(clips_dir),
    "--meta", str(OUT_DIR / "clips_meta.csv"),
    "--out", str(RESULTS),
    "--no-qa",               # remove to generate annotated QA videos
    "--first-move", "heel",  # alternatives: disp
], capture_output=True, text=True)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1500:])


## Step 4 — Build Per-Attempt Delivery Table

In [ ]:
# build_delivery.py — joins detector output with leads CSV
# Source:
#!/usr/bin/env python3
"""
Generalized per-attempt delivery-table builder (any runner / year).

Assembles <dir>/pitcher_delivery_<year>.csv (the schema delivery_velocity_*.py reads)
from:
  - <dir>/pilot_results.csv       full detector pass over the clips
  - /tmp/rec_<clip>.csv           optional per-clip 6s-cap recovery runs
  - <dir>/<leads>.csv             date / base / result join keys
  - a small reuse-fallback dict   (same-pitcher measured delivery) when a clip is
                                  un-measurable even after the cap

Usage:
  python3 cv_pilot/build_delivery.py --dir cv_pilot/Soto_2025 \
      --leads soto2025_leads.csv --year 2025 \
      [--recovered NAYLOR_x,NAYLOR_y] [--reuse play_id=delivery_s:note,...]

Selection per attempt:
  1. full pass usable=True            -> analysis_method=full
  2. else 6s-cap recovery usable=True -> analysis_method=capped_6s
  3. else reuse-fallback if provided  -> analysis_method=reused
  4. else keep failed number, usable=False
"""
import argparse, csv, os


def load(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))


def truthy(x):
    return str(x).strip().lower() == "true"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dir", required=True)
    ap.add_argument("--leads", required=True, help="leads filename inside --dir")
    ap.add_argument("--year", required=True)
    ap.add_argument("--recovered", default="", help="comma-separated clip_ids re-run with 6s cap")
    ap.add_argument("--reuse", default="",
                    help="comma-separated play_id=delivery_s:note entries")
    args = ap.parse_args()

    HERE = args.dir
    PILOT = os.path.join(HERE, "pilot_results.csv")
    LEADS = os.path.join(HERE, args.leads)
    OUT = os.path.join(HERE, f"pitcher_delivery_{args.year}.csv")

    recovered = [c for c in args.recovered.split(",") if c.strip()]
    reuse = {}
    for ent in args.reuse.split(","):
        ent = ent.strip()
        if not ent:
            continue
        pid, rest = ent.split("=", 1)
        ds, _, note = rest.partition(":")
        reuse[pid] = (float(ds), note or "reused")

    pilot = {r["play_id"]: r for r in load(PILOT)}
    leads = {r["play_id"]: r for r in load(LEADS)}

    rec = {}
    for c in recovered:
        p = f"/tmp/rec_{c}.csv"
        if os.path.exists(p):
            row = load(p)[0]
            rec[row["play_id"]] = row

    out_rows = []
    for play_id, L in leads.items():
        base = L["base"]; result = L["result"]
        event = ("Caught Stealing " if result == "CS" else "Stolen Base ") + base
        p = pilot.get(play_id, {})
        r = rec.get(play_id)
        delivery_s = usable = release_conf = method = note = None

        if p and truthy(p.get("usable")):
            delivery_s = p.get("delivery_s"); usable = True
            release_conf = p.get("release_kpt_conf"); method = "full"
        elif r and truthy(r.get("usable")):
            delivery_s = r.get("delivery_s"); usable = True
            release_conf = r.get("release_kpt_conf"); method = "capped_6s"
            note = "recovered via 6s analysis cap (broadcast follows runner)"
        elif play_id in reuse:
            delivery_s, note = reuse[play_id]
            usable = True; release_conf = ""; method = "reused"
        else:
            delivery_s = p.get("delivery_s", ""); usable = False
            release_conf = p.get("release_kpt_conf", ""); method = "full"
            note = "un-measurable; no same-pitcher fallback available"

        out_rows.append({
            "date": L["date"], "pitcher_id": L["pitcher_id"],
            "pitcher_name": L["pitcher_name"],
            "p_throws": p.get("p_throws", r.get("p_throws", "") if r else ""),
            "event": event, "catcher_name": L["catcher_name"],
            "clip_id": p.get("clip_id", r.get("clip_id", "") if r else ""),
            "play_id": play_id, "fps": p.get("fps", ""),
            "delivery_s": delivery_s, "usable": usable,
            "release_conf": release_conf, "analysis_method": method,
            "note": note or "",
        })

    out_rows.sort(key=lambda x: x["date"])
    cols = ["date", "pitcher_id", "pitcher_name", "p_throws", "event",
            "catcher_name", "clip_id", "play_id", "fps", "delivery_s",
            "usable", "release_conf", "analysis_method", "note"]
    with open(OUT, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader(); w.writerows(out_rows)

    n_us = sum(1 for r in out_rows if r["usable"])
    n_re = sum(1 for r in out_rows if r["analysis_method"] == "reused")
    n_cap = sum(1 for r in out_rows if r["analysis_method"] == "capped_6s")
    print(f"[write] {OUT}  ({len(out_rows)} attempts; {n_us} usable; "
          f"{n_cap} capped_6s; {n_re} reused)")
    for r in out_rows:
        if not r["usable"] or r["analysis_method"] != "full":
            print(f"  {r['date']} {r['pitcher_name']:22} {r['analysis_method']:9} "
                  f"d={r['delivery_s']} usable={r['usable']}  {r['note']}")


if __name__ == "__main__":
    main()



In [ ]:
result = subprocess.run([
    sys.executable, str(PIPE / "build_delivery.py"),
    "--dir", str(OUT_DIR),
    "--leads", str(LEADS),
    "--year", str(YEAR),
], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])


## Step 5 — Aggregate Delivery Times by Pitcher

In [ ]:
# aggregate_delivery.py — per-pitcher delivery time under stealing conditions
# Source:
#!/usr/bin/env python3
"""
aggregate_delivery.py  —  per-pitcher delivery time under stealing conditions
==============================================================================

Rolls the per-clip detector output (pilot_results.csv) up to a per-pitcher
delivery-time signature.  Every clip here is a real base-stealing pitch (the
runner broke on that delivery), so this is the per-pitcher mean *under stealing
conditions* — exactly what the Naylor model wants in place of the
LEAGUE_PITCHER_TTP = 1.30 constant.

Only `usable=True` clips feed the average (a clip is unusable when YOLO never
tracked the throwing hand, so its release is unreliable — see extract_delivery).

Output: cv_pilot/pitcher_delivery_sb.csv
    pitcher_id, pitcher_name, p_throws, n_clips, n_usable,
    mean_delivery_s, median_delivery_s, std_delivery_s, delivery_times
The keys (pitcher_id) join straight back to the Naylor model's DF_Attempts.
"""
from __future__ import annotations
from pathlib import Path
import pandas as pd

ROOT        = Path(__file__).resolve().parent
RESULTS_CSV = ROOT / "pilot_results.csv"
OUT_CSV     = ROOT / "pitcher_delivery_sb.csv"


def main():
    if not RESULTS_CSV.exists():
        raise SystemExit(f"{RESULTS_CSV} not found — run extract_delivery.py first")
    df = pd.read_csv(RESULTS_CSV)

    # need a pitcher key; fall back to a name parsed from clip_id if absent
    if "pitcher_id" not in df.columns:
        df["pitcher_id"] = pd.NA
    df["pitcher_key"] = df["pitcher_id"]
    miss = df["pitcher_key"].isna()
    if "pitcher_name" in df.columns:
        df.loc[miss, "pitcher_key"] = df.loc[miss, "pitcher_name"]

    usable = df[df.get("usable", False) == True].copy()      # noqa: E712

    rows = []
    for key, g in df.groupby("pitcher_key", dropna=False):
        gu = usable[usable["pitcher_key"] == key]
        times = gu["delivery_s"].dropna().tolist()
        name = (g["pitcher_name"].dropna().iloc[0]
                if "pitcher_name" in g and g["pitcher_name"].notna().any() else "")
        arm = (g["p_throws"].dropna().iloc[0]
               if "p_throws" in g and g["p_throws"].notna().any() else "")
        pid = (g["pitcher_id"].dropna().iloc[0]
               if "pitcher_id" in g and g["pitcher_id"].notna().any() else "")
        s = pd.Series(times)
        rows.append({
            "pitcher_id": pid,
            "pitcher_name": name,
            "p_throws": arm,
            "n_clips": int(len(g)),
            "n_usable": int(len(times)),
            "mean_delivery_s": round(s.mean(), 4) if len(times) else None,
            "median_delivery_s": round(s.median(), 4) if len(times) else None,
            "std_delivery_s": round(s.std(ddof=0), 4) if len(times) > 1 else None,
            "delivery_times": ";".join(f"{t:.3f}" for t in sorted(times)),
        })

    out = (pd.DataFrame(rows)
           .sort_values(["n_usable", "pitcher_name"], ascending=[False, True]))
    out.to_csv(OUT_CSV, index=False)

    n_meas = (out["n_usable"] > 0).sum()
    print(f"[write] {OUT_CSV}")
    print(f"   pitchers: {len(out)}  ·  with >=1 usable clip: {n_meas}")
    print(f"   pitchers with >=2 usable (std available): {(out['n_usable'] >= 2).sum()}")
    show = out[out["n_usable"] > 0][
        ["pitcher_name", "p_throws", "n_usable", "median_delivery_s",
         "std_delivery_s", "delivery_times"]]
    if len(show):
        print(show.to_string(index=False))


if __name__ == "__main__":
    main()



In [ ]:
from aggregate_delivery import main as aggregate_main
aggregate_main()


## Step 6 — Compute Runner Velocity Metric

In [ ]:
# delivery_velocity.py — avg_velocity_ftps = gain_to_release_ft / delivery_s
# Source:
#!/usr/bin/env python3
"""
Generalized delivery-window runner velocity metric (any runner / year).

    avg_velocity_ftps = gain_to_release_ft / delivery_s

gain_to_release_ft = lead_at_release_ft - lead_at_firstmove_ft   (Statcast lead data)
delivery_s         = CV-measured first-move -> ball-release time  (cv_pilot detector)

Joins leads ⟕ delivery on play_id (the Savant GUID), robust to duplicate
(date, pitcher) pairs. Replaces the per-folder delivery_velocity_<year>.py copies.

Usage:
  python3 cv_pilot/delivery_velocity.py --dir cv_pilot/Vladdy_2025 \
      --leads vladdy2025_leads.csv --year 2025 --runner "Vladimir Guerrero Jr."

Output: <dir>/delivery_velocity_<year>.csv
"""
import argparse, csv, os, statistics as st


def load_csv(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))


def fnum(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dir", required=True)
    ap.add_argument("--leads", required=True, help="leads filename inside --dir")
    ap.add_argument("--year", required=True)
    ap.add_argument("--runner", default="runner")
    args = ap.parse_args()

    HERE = args.dir
    LEADS = os.path.join(HERE, args.leads)
    DELIV = os.path.join(HERE, f"pitcher_delivery_{args.year}.csv")
    OUT = os.path.join(HERE, f"delivery_velocity_{args.year}.csv")

    leads = load_csv(LEADS)
    deliv = {r["play_id"]: r for r in load_csv(DELIV)}

    rows = []
    for L in leads:
        d = deliv.get(L["play_id"])
        if d is None:
            continue
        delivery_s = fnum(d.get("delivery_s"))
        usable = str(d.get("usable", "")).strip().lower() == "true"
        gain = fnum(L.get("gain_to_release_ft"))
        avg_vel = None
        if usable and delivery_s and delivery_s > 0 and gain is not None:
            avg_vel = round(gain / delivery_s, 3)
        rows.append({
            "date": L["date"], "pitcher_name": L["pitcher_name"],
            "pitcher_id": L["pitcher_id"], "p_throws": d.get("p_throws", ""),
            "base": L["base"], "result": L["result"], "run_value": L["run_value"],
            "lead_at_firstmove_ft": L["lead_at_firstmove_ft"],
            "lead_at_release_ft": L["lead_at_release_ft"],
            "gain_to_release_ft": L["gain_to_release_ft"],
            "delivery_s": d.get("delivery_s", ""), "release_conf": d.get("release_conf", ""),
            "usable": d.get("usable", ""), "analysis_method": d.get("analysis_method", ""),
            "play_id": L["play_id"],
            "avg_velocity_ftps": "" if avg_vel is None else avg_vel,
        })

    cols = ["date", "pitcher_name", "pitcher_id", "p_throws", "base", "result",
            "run_value", "lead_at_firstmove_ft", "lead_at_release_ft",
            "gain_to_release_ft", "delivery_s", "release_conf", "usable",
            "analysis_method", "play_id", "avg_velocity_ftps"]
    with open(OUT, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader(); w.writerows(rows)

    usable_rows = [r for r in rows if r["avg_velocity_ftps"] != ""]
    vals = [float(r["avg_velocity_ftps"]) for r in usable_rows]

    print("=" * 74)
    print(f"DELIVERY-WINDOW RUNNER VELOCITY  ({args.runner}, {args.year})")
    print("  avg_velocity_ftps = gain_to_release_ft / delivery_s")
    print("=" * 74)
    hdr = f"{'date':10} {'pitcher':22} {'res':4} {'gain':>5} {'deliv':>6} {'vel':>7} {'meth':>9}"
    print(hdr); print("-" * len(hdr))
    for r in sorted(rows, key=lambda x: (x["avg_velocity_ftps"] == "",
                    -(float(x["avg_velocity_ftps"]) if x["avg_velocity_ftps"] != "" else 0))):
        vel = r["avg_velocity_ftps"]
        vel_s = f"{float(vel):7.2f}" if vel != "" else "    n/a"
        dv = r["delivery_s"] or "n/a"
        g = fnum(r["gain_to_release_ft"])
        g_s = f"{g:5.1f}" if g is not None else "  n/a"
        print(f"{r['date']:10} {r['pitcher_name']:22} {r['result']:4} "
              f"{g_s} {str(dv):>6} {vel_s} {r['analysis_method']:>9}")
    print("-" * len(hdr))
    print(f"usable plays: {len(usable_rows)}/{len(rows)}")
    if vals:
        print(f"velocity ft/s  mean={st.mean(vals):.2f}  median={st.median(vals):.2f}  "
              f"min={min(vals):.2f}  max={max(vals):.2f}  "
              f"std={st.pstdev(vals):.2f}")

    sb = [float(r["avg_velocity_ftps"]) for r in usable_rows if r["result"] == "SB"]
    cs = [float(r["avg_velocity_ftps"]) for r in usable_rows if r["result"] == "CS"]
    print()
    if sb:
        print(f"  SB (n={len(sb)})  mean vel = {st.mean(sb):.2f} ft/s")
    if cs:
        print(f"  CS (n={len(cs)})  mean vel = {st.mean(cs):.2f} ft/s")
    print("=" * 74)
    print(f"wrote {OUT}")


if __name__ == "__main__":
    main()



In [ ]:
result = subprocess.run([
    sys.executable, str(PIPE / "delivery_velocity.py"),
    "--dir", str(OUT_DIR),
    "--leads", str(LEADS),
    "--year", str(YEAR),
    "--runner", RUNNER_NAME.capitalize(),
], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

import pandas as pd
vel_csv = OUT_DIR / f"delivery_velocity_{YEAR}.csv"
if vel_csv.exists():
    display(pd.read_csv(vel_csv).head(10))


## Step 7 — Statcast Reference Cross-Check

In [ ]:
# statcast_ref.py — compares CV delivery metric against Savant baselines
# Source:
#!/usr/bin/env python3
"""
Generalized Statcast base-stealing reference + head-to-head vs our CV metric
(any runner / year). Replaces the per-folder statcast_ref_<year>.py copies.

IMPORTANT — Savant data sourcing (year correctness):
  * The basestealing-run-value LEADERBOARD CSV export IGNORES its ?year= param, so it
    CANNOT be used for a past-season reference. The PER-ATTEMPT service
        /leaderboard/services/basestealing-running-game/{rid}?season_start=Y&season_end=Y
    DOES honor the year and is what <leads>.csv was built from, so the season
    SB-tracking anchors are computed from the leads CSV.
  * The sprint_speed leaderboard DOES honor ?min_season/?max_season.

Usage:
  python3 cv_pilot/statcast_ref.py --dir cv_pilot/Vladdy_2025 \
      --leads vladdy2025_leads.csv --year 2025 \
      --runner-id 665489 --runner "Vladimir Guerrero Jr."

Network: Savant has no sandbox DNS -> run with dangerouslyDisableSandbox.
Writes: <dir>/statcast_ref_<year>.csv, <dir>/metric_vs_statcast_<year>.csv
"""
import argparse, csv, io, os, statistics as st, requests

S = requests.Session(); S.headers.update({"User-Agent": "Mozilla/5.0"})


def getcsv(u):
    return list(csv.DictReader(io.StringIO(S.get(u, timeout=30).text.lstrip("﻿"))))


def fnum(x):
    try: return float(x)
    except (TypeError, ValueError): return None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dir", required=True)
    ap.add_argument("--leads", required=True)
    ap.add_argument("--year", required=True, type=int)
    ap.add_argument("--runner-id", required=True)
    ap.add_argument("--runner", default="runner")
    args = ap.parse_args()

    HERE = args.dir
    LEADS = os.path.join(HERE, args.leads)
    DV = os.path.join(HERE, f"delivery_velocity_{args.year}.csv")
    REF_OUT = os.path.join(HERE, f"statcast_ref_{args.year}.csv")
    H2H_OUT = os.path.join(HERE, f"metric_vs_statcast_{args.year}.csv")
    rid = str(args.runner_id)

    SPRINT = ("https://baseballsavant.mlb.com/leaderboard/sprint_speed"
              f"?attempts=1&min_season={args.year}&max_season={args.year}"
              "&position=&team=&csv=true")
    sprint = getcsv(SPRINT)
    s = next((r for r in sprint if r.get("player_id") == rid), None)

    leads = list(csv.DictReader(open(LEADS)))
    sb_rows = [r for r in leads if r["result"] == "SB"]
    cs_rows = [r for r in leads if r["result"] == "CS"]
    n_sb, n_cs = len(sb_rows), len(cs_rows)
    lead_rows = [r for r in leads if fnum(r["lead_at_firstmove_ft"]) is not None]
    fm_all = [fnum(r["lead_at_firstmove_ft"]) for r in lead_rows]
    rel_all = [fnum(r["lead_at_release_ft"]) for r in lead_rows]
    gain_all = [fnum(r["gain_to_release_ft"]) for r in lead_rows]
    fm_sb = [fnum(r["lead_at_firstmove_ft"]) for r in sb_rows if fnum(r["lead_at_firstmove_ft"]) is not None]
    rel_sb = [fnum(r["lead_at_release_ft"]) for r in sb_rows if fnum(r["lead_at_release_ft"]) is not None]

    ref = {
        "player": args.runner, "season": args.year,
        "sprint_speed_ftps": (s or {}).get("sprint_speed", ""),
        "hp_to_1b_s": (s or {}).get("hp_to_1b", ""),
        "competitive_runs": (s or {}).get("competitive_runs", ""),
        "n_tracked_attempts": len(leads), "n_sb": n_sb, "n_cs": n_cs,
        "sb_success_pct": round(100 * n_sb / (n_sb + n_cs), 1) if (n_sb + n_cs) else "",
        "steal_run_value": round(sum(fnum(r["run_value"]) for r in leads if fnum(r["run_value"]) is not None), 3),
        "primary_lead_ft": round(st.mean(fm_all), 2) if fm_all else "",
        "secondary_lead_ft": round(st.mean(rel_all), 2) if rel_all else "",
        "sec_minus_prim_lead_ft": round(st.mean(gain_all), 2) if gain_all else "",
        "primary_lead_sbx_ft": round(st.mean(fm_sb), 2) if fm_sb else "",
        "secondary_lead_sbx_ft": round(st.mean(rel_sb), 2) if rel_sb else "",
        "source_note": "SB anchors from per-attempt basestealing-running-game (year-correct); "
                       "sprint from sprint_speed leaderboard; leaderboard CSV year-param is broken",
    }
    with open(REF_OUT, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(ref)); w.writeheader(); w.writerow(ref)

    dv = list(csv.DictReader(open(DV)))
    usable = [r for r in dv if r["avg_velocity_ftps"] != ""]
    vel = [float(r["avg_velocity_ftps"]) for r in usable]
    lfm = [float(r["lead_at_firstmove_ft"]) for r in dv if r["lead_at_firstmove_ft"]]
    lrel = [float(r["lead_at_release_ft"]) for r in dv if r["lead_at_release_ft"]]
    sprint_v = fnum(ref["sprint_speed_ftps"])

    print("=" * 78)
    print(f"{args.runner.upper()} {args.year} — SB-TRACKING METRICS  (Statcast season + our CV metric)")
    print("=" * 78)
    print("SEASON STATCAST (year-correct; SB anchors from per-attempt service):")
    print(f"  sprint speed          {ref['sprint_speed_ftps']:>6} ft/s")
    print(f"  home-to-1B            {ref['hp_to_1b_s']:>6} s")
    print(f"  tracked attempts      {ref['n_tracked_attempts']:>6}")
    print(f"  primary lead (all)    {ref['primary_lead_ft']:>6} ft")
    print(f"  secondary lead (all)  {ref['secondary_lead_ft']:>6} ft")
    print(f"  sec-minus-prim gain   {ref['sec_minus_prim_lead_ft']:>6} ft")
    print(f"  SB / CS               {ref['n_sb']} / {ref['n_cs']}  "
          f"({ref['sb_success_pct']}% success)")
    print(f"  steal run value       {ref['steal_run_value']:>+6} runs")
    print()
    if vel:
        print("OUR CV METRIC (delivery-window velocity, per-attempt — varies by matchup):")
        print(f"  mean {st.mean(vel):.2f}  median {st.median(vel):.2f}  "
              f"range {min(vel):.2f}-{max(vel):.2f} ft/s   (n={len(vel)})")
        print()
        print("CROSS-VALIDATION  (same per-attempt rows, sanity check):")
        if lfm:
            print(f"  mean lead@firstmove = {st.mean(lfm):5.2f} ft   (= primary lead all = {ref['primary_lead_ft']})")
        if lrel:
            print(f"  mean lead@release   = {st.mean(lrel):5.2f} ft   (= secondary lead all = {ref['secondary_lead_ft']})")
        if sprint_v:
            print(f"  ratio our-vel / sprint-speed = {st.mean(vel)/sprint_v:.2f}  "
                  f"(~{100*st.mean(vel)/sprint_v:.0f}% of top speed during the delivery window)")
    print("=" * 78)

    rows = [(float(r["avg_velocity_ftps"]) if r["avg_velocity_ftps"] != "" else -1, r) for r in dv]
    rows.sort(key=lambda x: -x[0])
    out = []
    for v, r in rows:
        out.append({
            "date": r["date"], "pitcher_name": r["pitcher_name"], "result": r["result"],
            "avg_velocity_ftps": r["avg_velocity_ftps"],
            "lead_at_release_ft": r["lead_at_release_ft"],
            "delivery_s": r["delivery_s"], "analysis_method": r["analysis_method"],
            "runner_sprint_speed_ftps": ref["sprint_speed_ftps"],
            "runner_secondary_lead_sbx_ft": ref["secondary_lead_sbx_ft"],
        })
    if out:
        with open(H2H_OUT, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(out[0])); w.writeheader(); w.writerows(out)
    print(f"\nwrote {os.path.basename(REF_OUT)} + {os.path.basename(H2H_OUT)}")


if __name__ == "__main__":
    main()



In [ ]:
result = subprocess.run([
    sys.executable, str(PIPE / "statcast_ref.py"),
    "--dir", str(OUT_DIR),
    "--leads", str(LEADS),
    "--year", str(YEAR),
    "--runner-id", str(RUNNER_ID),
    "--runner", RUNNER_NAME.capitalize(),
], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])


## Step 8 — Evaluate Accuracy vs Manual Labels

In [ ]:
# evaluate.py — accuracy / repeatability / coverage + go-no-go verdict
# Gate: release error ≤ 2 frames (±66 ms), within-pitcher std ≤ 0.10 s, coverage ≥ 70%
# Source:
#!/usr/bin/env python3
"""
evaluate.py  —  Accuracy / repeatability / coverage + go-no-go verdict
======================================================================

Joins the automatic detector output (pilot_results.csv) against the manual
ground-truth labels (labels_manual.csv) and answers the only question the
pilot exists to answer:  is CV-derived pitch delivery time reliable enough
to trust a per-pitcher average?

Reports, and writes to PILOT_FINDINGS.md:
  • Accuracy     — release & first-move frame error (frames + ms), bias, MAE;
                   a fitted constant release offset (peak-vs-true correction).
  • Repeatability— within-pitcher delivery std across that pitcher's clips.
  • Coverage     — % of clips that yield a usable auto measurement.
  • Cross-check  — per-pitcher mean vs. a scout-TTP-minus-flight sanity band.
  • Verdict      — GO / NO-GO against the gate, reported honestly either way.

Gate (from the plan):
    release error  ≤ 2 frames   (≈ ±66 ms @ 30 fps)
    within-pitcher delivery std ≤ 0.10 s
    coverage       ≥ 70 %

Usage:
    python3 cv_pilot/evaluate.py
"""

from __future__ import annotations
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd

ROOT        = Path(__file__).resolve().parent
RESULTS_CSV = ROOT / "pilot_results.csv"
LABELS_CSV  = ROOT / "labels_manual.csv"
FINDINGS_MD = ROOT / "PILOT_FINDINGS.md"

# Gate thresholds
GATE_REL_FRAMES = 2.0
GATE_WITHIN_STD = 0.10
GATE_COVERAGE   = 0.70

# Cross-check: typical first-move->release for MLB ~ 0.95-1.45 s; flight ~0.40 s.
SANITY_BAND = (0.85, 1.55)


def fmt(x, nd=3):
    return "n/a" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{x:.{nd}f}"


def main():
    if not RESULTS_CSV.exists():
        raise SystemExit(f"Missing {RESULTS_CSV}. Run extract_delivery.py first.")
    res = pd.read_csv(RESULTS_CSV)

    n_total = len(res)
    usable = res[res.get("usable", False) == True] if "usable" in res else res
    coverage = len(usable) / n_total if n_total else 0.0

    lines = []
    lines.append("# Pitch Delivery CV Pilot — Findings")
    lines.append("")
    lines.append(f"_Generated {date.today().isoformat()}_")
    lines.append("")
    lines.append("## Coverage")
    lines.append("")
    lines.append(f"- Clips processed: **{n_total}**")
    lines.append(f"- Usable auto measurements (events found pre-cut, in-band, "
                 f"confident): **{len(usable)}**")
    lines.append(f"- **Coverage = {coverage*100:.1f}%**  "
                 f"(gate ≥ {GATE_COVERAGE*100:.0f}%)")
    lines.append("")

    # ── Accuracy vs manual labels ───────────────────────────────────────────
    rel_mae = fm_mae = rel_bias = rel_off = None
    have_labels = LABELS_CSV.exists()
    merged = pd.DataFrame()
    if have_labels:
        lab = pd.read_csv(LABELS_CSV)
        merged = res.merge(lab, on="clip_id", how="inner", suffixes=("", "_lab"))
        merged = merged.dropna(subset=["release_frame", "manual_release_frame"])

    lines.append("## Accuracy (auto vs. manual ground truth)")
    lines.append("")
    if not have_labels or merged.empty:
        lines.append("_No manual labels yet — run `label_tool.py` to build "
                     "ground truth, then re-run this script._")
        lines.append("")
    else:
        rel_err = merged["release_frame"] - merged["manual_release_frame"]
        fm_err  = merged["first_move_frame"] - merged["manual_first_move_frame"]
        rel_mae  = float(rel_err.abs().mean())
        rel_bias = float(rel_err.mean())
        fm_mae   = float(fm_err.abs().mean())
        rel_off  = rel_bias                       # constant correction to apply
        fps_mean = float(merged["fps"].mean())
        ms = 1000.0 / fps_mean

        lines.append(f"- n labeled clips: **{len(merged)}**  (mean {fps_mean:.1f} fps)")
        lines.append(f"- Release frame error — MAE **{rel_mae:.2f} frames "
                     f"(≈ {rel_mae*ms:.0f} ms)**, bias {rel_bias:+.2f} frames")
        lines.append(f"- First-move frame error — MAE **{fm_mae:.2f} frames "
                     f"(≈ {fm_mae*ms:.0f} ms)**")
        lines.append(f"- Fitted constant release offset (subtract from auto): "
                     f"**{rel_off:+.2f} frames** — applying it would reduce "
                     f"systematic bias.")
        # after constant-offset correction
        corr_mae = float((rel_err - rel_off).abs().mean())
        lines.append(f"- Release MAE after offset correction: **{corr_mae:.2f} frames**")
        lines.append("")
        lines.append("| clip | auto rel | manual rel | err (frames) | auto delivery_s | manual delivery_s |")
        lines.append("|---|---|---|---|---|---|")
        for _, r in merged.iterrows():
            lines.append(
                f"| {r['clip_id']} | {fmt(r['release_frame'],1)} | "
                f"{fmt(r['manual_release_frame'],1)} | "
                f"{fmt(r['release_frame']-r['manual_release_frame'],1)} | "
                f"{fmt(r.get('delivery_s'),3)} | "
                f"{fmt(r.get('manual_delivery_s'),3)} |")
        lines.append("")

    # ── Delivery-time accuracy (the production metric) ──────────────────────
    # release_frame MAE above is corrupted by multi-segment ABSOLUTE indexing
    # (the detector may lock onto a later replay/segment, so its frame index is
    # offset from the manually labelled first-broadcast view).  delivery_s is
    # segment-invariant, so the honest accuracy number is the delivery_s error —
    # and only on the `usable` clips the pipeline actually feeds downstream.
    lines.append("## Delivery-time accuracy (delivery_s, the production metric)")
    lines.append("")
    if not merged.empty and "delivery_s" in merged and "manual_delivery_s" in merged:
        dd = merged.dropna(subset=["delivery_s", "manual_delivery_s"]).copy()
        dd["ae"] = (dd["delivery_s"] - dd["manual_delivery_s"]).abs()
        u = dd[dd.get("usable", False) == True] if "usable" in dd else dd
        for tag, sub in [("ALL labelled+detected", dd), ("usable only", u)]:
            if len(sub):
                bias = float((sub["delivery_s"] - sub["manual_delivery_s"]).mean())
                lines.append(
                    f"- {tag}: n={len(sub)}, **MAE {sub['ae'].mean():.3f}s** "
                    f"(median {sub['ae'].median():.3f}s), bias {bias:+.3f}s, "
                    f"|err|>0.30s on {int((sub['ae']>0.30).sum())}")
        lines.append("")

    # ── Repeatability (within-pitcher std) ──────────────────────────────────
    lines.append("## Repeatability (within-pitcher delivery std)")
    lines.append("")
    worst_std = None
    key = "pitcher_name" if "pitcher_name" in usable.columns else "pitcher_id"
    if key in usable.columns and "delivery_s" in usable.columns and len(usable):
        grp = (usable.dropna(subset=["delivery_s"])
                     .groupby(key)["delivery_s"]
                     .agg(["count", "mean", "std"]))
        if not grp.empty:
            lines.append(f"| pitcher | n | mean delivery_s | std (s) |")
            lines.append("|---|---|---|---|")
            for name, r in grp.iterrows():
                lines.append(f"| {name} | {int(r['count'])} | "
                             f"{fmt(r['mean'],3)} | {fmt(r['std'],3)} |")
            multi = grp[grp["count"] >= 2]
            if not multi.empty:
                worst_std = float(multi["std"].max())
                lines.append("")
                lines.append(f"- Worst within-pitcher std (n≥2): "
                             f"**{worst_std:.3f} s**  (gate ≤ {GATE_WITHIN_STD:.2f} s)")
            lines.append("")
    else:
        lines.append("_Need ≥2 usable clips per pitcher to assess repeatability._")
        lines.append("")

    # ── External sanity band ────────────────────────────────────────────────
    lines.append("## External sanity check")
    lines.append("")
    if "delivery_s" in usable.columns and len(usable):
        d = usable["delivery_s"].dropna()
        in_band = ((d >= SANITY_BAND[0]) & (d <= SANITY_BAND[1])).mean() if len(d) else 0
        lines.append(f"- Usable deliveries within plausible MLB band "
                     f"{SANITY_BAND[0]:.2f}–{SANITY_BAND[1]:.2f}s: "
                     f"**{in_band*100:.0f}%**")
        lines.append(f"- Median usable delivery_s: **{fmt(d.median(),3)}** "
                     f"(scout TTP ≈ delivery + ~0.40s ball-flight)")
        lines.append("")

    # ── Verdict ─────────────────────────────────────────────────────────────
    pass_cov = coverage >= GATE_COVERAGE
    pass_acc = (rel_mae is not None) and (rel_mae <= GATE_REL_FRAMES)
    pass_rep = (worst_std is not None) and (worst_std <= GATE_WITHIN_STD)
    checks = []
    checks.append(("coverage ≥ 70%", pass_cov, f"{coverage*100:.1f}%"))
    checks.append(("release MAE ≤ 2 frames", pass_acc,
                   fmt(rel_mae, 2) + " frames" if rel_mae is not None else "no labels"))
    checks.append(("within-pitcher std ≤ 0.10s", pass_rep,
                   fmt(worst_std, 3) + " s" if worst_std is not None else "insufficient n"))

    go = pass_cov and (pass_acc is True) and (pass_rep is True)
    verdict = "GO" if go else ("NO-GO" if (rel_mae is not None and worst_std is not None)
                               else "INCOMPLETE (need manual labels / more clips)")

    lines.append("## Verdict")
    lines.append("")
    for name, ok, val in checks:
        mark = "✅" if ok else ("❌" if ok is False else "⚠️")
        lines.append(f"- {mark} {name}: {val}")
    lines.append("")
    lines.append(f"### → **{verdict}**")
    lines.append("")
    if verdict == "GO":
        lines.append("Extraction is reliable enough to build per-pitcher averages. "
                     "Proceed to the scaling step (fetch_clips.py → DF_PitcherDelivery.csv → "
                     "replace LEAGUE_PITCHER_TTP in v6_explore.py).")
    elif verdict.startswith("NO-GO"):
        lines.append("Extraction is **not** reliable enough at this resolution. "
                     "The model keeps the LEAGUE_PITCHER_TTP=1.30 constant unchanged. "
                     "This is a clean negative result, not a bug — see the error "
                     "numbers above. Options: higher-fps clips, better release model, "
                     "or accept per-pitcher means with wider uncertainty.")
    else:
        lines.append("Run `label_tool.py` on all clips and/or add more clips per "
                     "pitcher, then re-run `evaluate.py` for a final verdict.")
    lines.append("")

    FINDINGS_MD.write_text("\n".join(lines))
    print("\n".join(lines))
    print(f"\n[write] {FINDINGS_MD}")


if __name__ == "__main__":
    main()



In [ ]:
result = subprocess.run([
    sys.executable, str(PIPE / "evaluate.py"),
], capture_output=True, text=True, cwd=str(OUT_DIR))
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])


## Shortcut — run_runner.py (Steps 1–3 in one command)

Instead of running steps 1–3 individually, `run_runner.py` chains them:
```bash
python3 "CV Detection Pipeline/run_runner.py" 647304 naylor 2025
python3 "CV Detection Pipeline/run_runner.py" 665742 soto   2025 --no-detect
```


In [ ]:
# run_runner.py — one-command orchestrator for leads → clips → delivery
# Source:
#!/usr/bin/env python3
"""
run_runner.py  —  One-command per-runner pipeline (leads -> clips -> delivery)
=============================================================================

Takes a single (runner_id, name_tag, year) — typically a row that
discover_runners.py surfaced — and runs the whole hands-off chain that used to
be manual hovering + clicking + per-folder scripting:

    1. fetch_leads.py      runner_id year  -> <dir>/<tag><year>_leads.csv
                                              <dir>/<tag><year>_targets.csv
    2. fetch_clips.py      --targets ...    -> <dir>/clips/*.mp4 + clips_meta.csv
    3. extract_delivery.py --clips-dir ...  -> <dir>/pilot_results.csv  (heel default)

Then the existing delivery_velocity.py / statcast_ref.py can be pointed at <dir>.

Usage
-----
  python3 cv_pilot/run_runner.py 647304 naylor 2025
  python3 cv_pilot/run_runner.py 665742 soto 2025 --dir cv_pilot/Soto_2025 --no-detect
  python3 cv_pilot/run_runner.py 592518 machado 2025 --dry-run   # print the plan only

Network stages (leads, clips) need dangerouslyDisableSandbox.  The detector stage
is CPU/MPS-heavy (~tens of seconds per clip); pass --no-detect to stop after the
clips download and run extract_delivery.py yourself later.
"""
from __future__ import annotations
import argparse
import subprocess
import sys
from pathlib import Path

HERE = Path(__file__).resolve().parent


def run(cmd: list[str], dry: bool) -> int:
    print("   $", " ".join(str(c) for c in cmd))
    if dry:
        return 0
    return subprocess.run(cmd, check=False).returncode


def main():
    ap = argparse.ArgumentParser(description="Per-runner leads->clips->delivery pipeline")
    ap.add_argument("runner_id", type=int)
    ap.add_argument("name_tag", help="lowercase tag for file/clip naming, e.g. 'naylor'")
    ap.add_argument("year", type=int)
    ap.add_argument("--dir", default=None, help="output dir (default cv_pilot/<Tag>_<year>)")
    ap.add_argument("--no-clips", action="store_true", help="stop after leads")
    ap.add_argument("--no-detect", action="store_true", help="stop after clips (skip detector)")
    ap.add_argument("--first-move", choices=["disp", "heel"], default="heel",
                    help="detector first-move cue (default heel)")
    ap.add_argument("--dry-run", action="store_true", help="print the commands, run nothing")
    args = ap.parse_args()

    tag, y = args.name_tag, args.year
    d = Path(args.dir) if args.dir else (HERE / f"{tag.capitalize()}_{y}")
    leads = d / f"{tag}{y}_leads.csv"
    targets = d / f"{tag}{y}_targets.csv"
    py = sys.executable

    print(f"[run] {tag} ({args.runner_id}) {y} -> {d}")

    # 1) leads
    rc = run([py, str(HERE / "fetch_leads.py"), str(args.runner_id), str(y),
              "--runner-name", tag, "--out", str(leads), "--targets-out", str(targets)],
             args.dry_run)
    if rc != 0 and not args.dry_run:
        sys.exit(f"fetch_leads failed (rc={rc})")
    if args.no_clips:
        return

    # 2) clips
    rc = run([py, str(HERE / "fetch_clips.py"), "--targets", str(targets),
              "--out-dir", str(d), "--download"], args.dry_run)
    if rc != 0 and not args.dry_run:
        sys.exit(f"fetch_clips failed (rc={rc})")
    if args.no_detect:
        return

    # 3) detector (heel default)
    rc = run([py, str(HERE / "extract_delivery.py"),
              "--clips-dir", str(d / "clips"), "--meta", str(d / "clips_meta.csv"),
              "--out", str(d / "pilot_results.csv"), "--no-qa",
              "--first-move", args.first_move], args.dry_run)
    if rc != 0 and not args.dry_run:
        sys.exit(f"extract_delivery failed (rc={rc})")

    print(f"[done] {tag} {y}: leads + clips + pilot_results.csv under {d}")


if __name__ == "__main__":
    main()



## build_pitcher_delivery_table.py — league-wide pitcher table

In [ ]:
# build_pitcher_delivery_table.py — aggregates all runner-season pilot_results
# into a single pitcher delivery table for the Naylor Model
# Source:
#!/usr/bin/env python3
"""
Consolidate every CV-measured pitcher delivery time into one lookup table:
    data/DF_PitcherDelivery.csv  ->  pitcher_id, n_clips, median_delivery_s

This is the join table v6_explore.py uses to replace the constant LEAGUE_PITCHER_TTP
(1.30 s) with a measured per-pitcher first-move->release time when we have one.

Sources (all under cv_pilot/):
  - pitcher_delivery_sb.csv               20-pitcher steal-condition pool (raw times)
  - Naylor_2025/pitcher_delivery_2025.csv per-attempt (usable, NOT reused)
  - Naylor_2026/pitcher_delivery_2026.csv per-attempt (usable)
  - Soto_2025/pitcher_delivery_2025.csv   per-attempt (usable, NOT reused)
  - Soto_2026/pitcher_delivery_2026.csv   per-attempt (usable)
  - {Vladdy,Yandy,Torres,Bichette}_2025/pitcher_delivery_2025.csv  per-attempt
  - Naylor_2024  leads + /tmp detector rows (3 attempts)

We pool independent measurements per pitcher_id and take the MEDIAN. Rows flagged
analysis_method == 'reused' are skipped (they are copies of another measurement and
would double-count).
"""
import csv, os, statistics as st

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
OUT = os.path.join(ROOT, "data", "DF_PitcherDelivery.csv")

times = {}   # pitcher_id(int) -> list[float]


def add(pid, val):
    try:
        pid = int(float(pid)); val = float(val)
    except (TypeError, ValueError):
        return
    if val > 0:
        times.setdefault(pid, []).append(val)


# 1) 20-pitcher pool (raw per-clip times in 'delivery_times', ';'-joined)
pool = os.path.join(HERE, "pitcher_delivery_sb.csv")
if os.path.exists(pool):
    for r in csv.DictReader(open(pool)):
        pid = r.get("pitcher_id")
        for t in (r.get("delivery_times") or "").split(";"):
            if t.strip():
                add(pid, t)

# 2) Naylor + Soto per-attempt delivery tables (skip reused)
for rel in ("Naylor_2025/pitcher_delivery_2025.csv",
            "Naylor_2026/pitcher_delivery_2026.csv",
            "Soto_2025/pitcher_delivery_2025.csv",
            "Soto_2026/pitcher_delivery_2026.csv",
            "Vladdy_2025/pitcher_delivery_2025.csv",
            "Yandy_2025/pitcher_delivery_2025.csv",
            "Torres_2025/pitcher_delivery_2025.csv",
            "Bichette_2025/pitcher_delivery_2025.csv",
            "Freeman_2025/pitcher_delivery_2025.csv",
            "Gurriel_2025/pitcher_delivery_2025.csv",
            "Young_2025/pitcher_delivery_2025.csv",
            "Holliday_2025/pitcher_delivery_2025.csv"):
    p = os.path.join(HERE, rel)
    if not os.path.exists(p):
        continue
    for r in csv.DictReader(open(p)):
        if str(r.get("usable", "")).strip().lower() != "true":
            continue
        if (r.get("analysis_method") or "") == "reused":
            continue
        add(r.get("pitcher_id"), r.get("delivery_s"))

# 3) Naylor 2024 (leads carry pitcher_id; deliveries in /tmp/d2024_*.csv)
leads24 = os.path.join(HERE, "Naylor_2024", "naylor2024_leads.csv")
if os.path.exists(leads24):
    by_play = {r["play_id"]: r for r in csv.DictReader(open(leads24))}
    for c in ("NAYLOR_d21c5fbb_muller_L", "NAYLOR_977c39d5_sale_L",
              "NAYLOR_ea0116d0_hernandez_R"):
        f = f"/tmp/d2024_{c}.csv"
        if os.path.exists(f):
            d = list(csv.DictReader(open(f)))[0]
            L = by_play.get(d.get("play_id"))
            if L and str(d.get("usable", "")).strip().lower() == "true":
                add(L["pitcher_id"], d.get("delivery_s"))

os.makedirs(os.path.dirname(OUT), exist_ok=True)
rows = []
for pid, vals in sorted(times.items()):
    rows.append({"pitcher_id": pid, "n_clips": len(vals),
                 "median_delivery_s": round(st.median(vals), 4)})
with open(OUT, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["pitcher_id", "n_clips", "median_delivery_s"])
    w.writeheader(); w.writerows(rows)

allv = [v for vs in times.values() for v in vs]
print(f"[write] {OUT}")
print(f"  {len(rows)} pitchers measured  ({len(allv)} total clips)")
print(f"  median delivery across pitchers = "
      f"{st.median([r['median_delivery_s'] for r in rows]):.3f} s  "
      f"(constant fallback was 1.300)")

